# 💧 WaterSec AI Agent — Hackathon WaterSec

> **Auteur** : Équipe WaterSec Hackathon  
> **Architecture** : RAG + ML multi-modèle + LangGraph + Groq LLM (gratuit)  
> **Données** : CustomerA (Offices) · CustomerB (Bloc sanitaire) · CustomerC (Résidentiel) · Gym (Cabines douche)

## Organisation du notebook
1. Exploration des données et préparation
2. Enrichissement des profils et features
3. Modèles de détection et scoring
4. Orchestration conversationnelle LLM
5. Export et visualisation des résultats

## 🏗️ Architecture du pipeline
```
                    ┌─────────────────────────────────┐
                    │      Requête Langage Naturel     │
                    └──────────────┬──────────────────┘
                                   │
                    ┌──────────────▼──────────────────┐
                    │    LangGraph Orchestrateur       │
                    │    (router intelligent)          │
                    └──┬──────┬─────────┬──────────┬──┘
                       │      │         │          │
                  Query  Analysis  Anomaly  Recommendation
                       │      │         │          │
          ┌────────────▼──────▼─────────▼──────────▼────────────┐
          │                 Couche Données                       │
          │  pandas SQL-like │ ChromaDB RAG │ ML Models          │
          │  (query_data)    │ (semantic)   │ (IsoForest+Stats)  │
          └────────────────────────────────────────────────────┘
                                   │
                    ┌──────────────▼──────────────────┐
                    │  Groq LLM (llama-3.1-8b-instant) │
                    │  Raisonnement + Narration        │
                    └─────────────────────────────────┘
```

## Fonctionnalités clés
| # | Fonctionnalité | Description |
|---|----------------|-------------|
| 1 | **Profils horaires WDI** | Water Demand Index : courbe de demande journalière normalisée par profil client |
| 2 | **Score d'efficacité hydrique** | Benchmark inter-client pondéré par type d'usage |

| 3 | **Détection de fuites continues** | Algorithme basé sur la persistance des débits nocturnes |

| 4 | **Analyse comportementale séquentielle** | Détection de chaînes d'événements (Flush→Sink→Tap) avec Markov || 8 | **Routeur LLM amélioré** | Classification d'intention via Groq au lieu de regex |

| 5 | **Prévision Prophet 7j** | Forecasting de la consommation à 7 jours par device || 7 | **Mémoire conversationnelle** | L'agent retient les 5 derniers échanges |
| 6 | **Dashboard KPI HTML exportable** | Rapport visuel interactif avec Plotly |

---
## 0. 📦 Installation des dépendances
> **⚠️ Exécuter une seule fois — redémarrer le kernel après si c'est la première fois**

In [18]:
import os

# ⚠️  Première installation uniquement — décommenter et exécuter, puis re-commenter
# from IPython import get_ipython
# get_ipython().kernel.do_shutdown(restart=True)

print("✅ Si c'est la première installation, redémarrez manuellement le kernel (Menu > Kernel > Restart).")


✅ Si c'est la première installation, redémarrez manuellement le kernel (Menu > Kernel > Restart).


In [19]:
# ── Installation principale ──────────────────────────────────────────────────
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langgraph==0.2.14 \
    chromadb==0.5.5 \
    sentence-transformers==3.0.1 \
    groq \
    prophet==1.1.5 \
    plotly==5.22.0 \
    kaleido==0.2.1 \
    requests \

# ── Mise à jour des librairies scientifiques ─────────────────────────────────
!pip install -q -U numpy==1.26.4 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.1

print("✅ Dépendances installées — redémarrer le kernel si c'est la première installation")

✅ Dépendances installées — redémarrer le kernel si c'est la première installation


In [20]:
# Prophet sera importé après installation (voir cellule Section 10)
print("✅ Cellule réservée — import Prophet effectué en Section 10.")

✅ Cellule réservée — import Prophet effectué en Section 10.


In [13]:
!pip uninstall -y prophet cmdstanpy -q

!pip install -q \
    cmdstanpy==1.2.4 \
    prophet==1.1.6

print("✅ Prophet corrigé — redémarrer le kernel")

✅ Prophet corrigé — redémarrer le kernel


In [ ]:
import os
os.kill(os.getpid(), 9)

In [16]:
import os

# ⚠️  Première installation uniquement — décommenter et exécuter, puis re-commenter
# from IPython import get_ipython
# get_ipython().kernel.do_shutdown(restart=True)

print("✅ Si c'est la première installation, redémarrez manuellement le kernel (Menu > Kernel > Restart).")


✅ Si c'est la première installation, redémarrez manuellement le kernel (Menu > Kernel > Restart).


In [1]:
import numpy as np
import pandas as pd
import scipy
import chromadb

print(scipy.__version__)
print(np.__version__)
print(pd.__version__)

1.13.1
1.26.4
2.2.2


---
## 1. ⚙️ Imports & Configuration

> **Configuration Groq** : Gratuit, rapide (~200 tok/s), sans téléchargement de modèle local.
> 1. Créer un compte sur https://console.groq.com  
> 2. Générer une clé API (`gsk_...`)  
> 3. Coller la clé dans `GROQ_API_KEY` ci-dessous

In [ ]:
import os, json, warnings, textwrap, time, re, requests
from collections import defaultdict, deque
from typing import TypedDict, Optional, List, Dict, Any

# Uninstall and reinstall numpy and pandas to resolve potential binary incompatibility
!pip uninstall -y numpy pandas
!pip install numpy==1.26.4 pandas==2.2.2

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.signal import find_peaks
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ─── Chemins datasets ────────────────────────────────────────────────────────
from pathlib import Path
root_dir = Path.cwd() if Path.cwd().name != 'notebooks' else Path.cwd().parent
DATA_DIR = (root_dir / 'data').resolve()
print('DATA_DIR =', DATA_DIR)

# ─── LLM Config — Groq API (100 % gratuit) ───────────────────────────────────
GROQ_API_KEY = os.environ.get('GROQ_API_KEY', 'make yours')  # ← Définir la variable d'env GROQ_API_KEY ou coller votre clé ici
GROQ_MODEL   = 'llama-3.3-70b-versatile'   # Gratuit, ~200 tok/s
# Alternatives gratuites :
#   'mixtral-8x7b-32768'      — meilleur raisonnement, contexte 32k
#   'llama-3.3-70b-versatile' — plus puissant (limite quotidienne)
#   'gemma2-9b-it'            — bon équilibre
LLM_MODE     = 'groq' if GROQ_API_KEY else 'mock'  # Bascule auto en mock si pas de clé

# ─── Localisation du site (pour météo et horaires de prière) ─────────────────
# Modifier ces valeurs selon la localisation réelle des sites WaterSec
SITE_LAT     = 36.8065   # Latitude  (Tunis par défaut — à adapter)
SITE_LON     = 10.1815   # Longitude
SITE_CITY    = 'Tunis'
SITE_COUNTRY = 'Tunisia'
PRAYER_METHOD = 3         # Muslim World League — méthode de calcul des prières

# ─── Palette couleurs par client ─────────────────────────────────────────────
COLORS = {'A': '#2196F3', 'B': '#4CAF50', 'C': '#FF5722', 'Gym': '#9C27B0'}

# ─── Seuils domaine eau ───────────────────────────────────────────────────────
# Valeurs de référence issues des normes européennes de consommation d'eau
WATER_BENCHMARKS = {
    'shower_L_per_use'    : 60,    # Douche standard : 60 L/utilisation
    'toilet_flush_L'      : 6,     # Chasse eau économique : 6 L
    'sink_use_L'          : 2,     # Robinet lavabo : ~2 L/utilisation
    'night_flow_threshold': 0.1,   # Fuite : débit nocturne > 0.1 L/min
    'leak_persistence_min': 30,    # Durée minimale d'une fuite (minutes)
    'wudu_L_per_use'      : 1.5,   # 🆕 Ablution (وضوء) : ~1.5 L/utilisation
}

print('✅ Configuration prête | LLM_MODE =', LLM_MODE, '| Modèle =', GROQ_MODEL)
print('📊 Benchmarks eau chargés :', WATER_BENCHMARKS)

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.3.3
Uninstalling pandas-2.3.3:
  Successfully uninstalled pandas-2.3.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 892.6 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 13.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 24.4 MB/s eta 0:00:0000:0100:01


---
## 2. 📊 Couche Données — Chargement & Unification

**Défis de ce dataset :**
- 4 sources avec des formats différents (séparateurs, colonnes)
- Timestamps corrompus / epoch Unix mélangés avec ISO8601
- Unités en mililitres, pas en litres
- Valeurs aberrantes (overflow uint32 = 4 294 967 295)

In [29]:
def safe_parse_timestamp(series: pd.Series) -> pd.Series:
    """
    Parsing robuste des timestamps.
    Gère : ISO8601, timestamps Unix (secondes), valeurs nulles/corrompues.
    Filtre les dates hors plage 2022-2027 (données spurieuses).
    """
    # Tentative 1 : format ISO8601 standard
    parsed = pd.to_datetime(series, utc=True, errors='coerce')

    # Tentative 2 : timestamps Unix valides (entre 2020 et 2030)
    mask_bad   = parsed.isna()
    numeric    = pd.to_numeric(series[mask_bad], errors='coerce')
    unix_valid = (numeric > 1_577_836_800) & (numeric < 1_893_456_000)  # 2020–2030
    parsed[mask_bad & unix_valid.reindex(series.index, fill_value=False)] = pd.to_datetime(
        numeric[unix_valid], unit='s', utc=True
    )

    # Filtre de cohérence temporelle
    valid_range = (parsed >= '2022-01-01') & (parsed <= '2027-01-01')
    parsed[~valid_range] = pd.NaT
    return parsed


def load_customer(path: str, sep: str, customer_name: str,
                  category: str, sub_category: str = None) -> pd.DataFrame:
    """Charge et normalise un CSV client vers le schéma unifié."""
    df = pd.read_csv(path, sep=sep)

    # Normalisation des noms de colonnes (Gym utilise 'data.xxx')
    df = df.rename(columns={
        'data.consumption' : 'data_consumption',
        'data.time'        : 'data_time',
        'data.period'      : 'data_period',
    })

    df['customer'] = customer_name
    if 'main_category_name' not in df.columns:
        df['main_category_name'] = category
    if sub_category and 'sub_category_name' not in df.columns:
        df['sub_category_name'] = sub_category
    if 'sub_category_name' not in df.columns:
        df['sub_category_name'] = None
    if 'tag' not in df.columns:
        df['tag'] = 'cold'

    return df[['customer', 'device', 'data_consumption', 'data_time',
               'data_period', 'tag', 'main_category_name', 'sub_category_name']]


def build_unified_df(data_dir: str) -> pd.DataFrame:
    """Pipeline complet de construction du dataset unifié."""
    # ── 1. Chargement des 4 sources ──────────────────────────────────────────
    dfs = [
        load_customer(f'{data_dir}/customerA_consumption.csv', ';', 'A',   'Offices'),
        load_customer(f'{data_dir}/customerB_consumption.csv', ';', 'B',   'Bloc sanitaire'),
        load_customer(f'{data_dir}/customerC_consumption.csv', ';', 'C',   'Bathroom'),
        load_customer(f'{data_dir}/gym_consumption_data.csv',  ',', 'Gym', 'Gym Shower', 'Shower'),
    ]
    df = pd.concat(dfs, ignore_index=True)

    # ── 2. Parsing des timestamps (robuste) ──────────────────────────────────
    df['timestamp'] = safe_parse_timestamp(df['data_time'])
    df = df[df['timestamp'].notna()].copy()

    # ── 3. Nettoyage des consommations ───────────────────────────────────────
    df['data_consumption'] = (
        pd.to_numeric(df['data_consumption'], errors='coerce')
        .clip(upper=500_000)   # Suppression overflow uint32
        .clip(lower=0)         # Pas de consommation négative
    )
    df['data_period'] = pd.to_numeric(df['data_period'], errors='coerce').clip(lower=0)

    # ── 4. Suppression des doublons exacts ───────────────────────────────────
    df = df.drop_duplicates(subset=['device', 'data_time', 'data_consumption'])

    # ── 5. Colonnes dérivées ─────────────────────────────────────────────────
    df['consumption_L']  = df['data_consumption'] / 1000          # mL → L
    df['flow_rate_Lmin'] = np.where(
        df['data_period'] > 0,
        df['data_consumption'] / df['data_period'] * 60 / 1000,   # mL/s → L/min
        np.nan
    )
    df['hour']      = df['timestamp'].dt.hour
    df['dayofweek'] = df['timestamp'].dt.dayofweek
    df['date']      = df['timestamp'].dt.date
    df['month']     = df['timestamp'].dt.month
    df['week']      = df['timestamp'].dt.isocalendar().week.astype(int)
    df['is_weekend']= df['dayofweek'].isin([5, 6]).astype(int)
    df['is_night']  = ((df['hour'] >= 23) | (df['hour'] <= 5)).astype(int)

    # ── 6. 🆕 Numérotation des cabines Gym ───────────────────────────────────
    # On mappe les UUID de devices Gym vers des labels lisibles (Cabin 1-8)
    # et on extrait Hot/Cold depuis le tag
    gym_devices = df[df['customer'] == 'Gym']['device'].unique()
    gym_device_map = {d: f'Cabin {i+1}' for i, d in enumerate(sorted(gym_devices))}
    df['cabin'] = df.apply(
        lambda r: gym_device_map.get(r['device'], '') if r['customer'] == 'Gym' else '', axis=1
    )

    df = df.sort_values(['customer', 'device', 'timestamp']).reset_index(drop=True)
    return df


# ── Chargement ────────────────────────────────────────────────────────────────
df = build_unified_df(DATA_DIR)

print(f'✅ Dataset unifié : {df.shape[0]:,} enregistrements')
print(f'   Doublons supprimés, timestamps nettoyés, débit calculé')
print()
display(df.groupby('customer').agg(
    enregistrements=('device', 'count'),
    devices_uniques=('device', 'nunique'),
    debut=('timestamp', 'min'),
    fin=('timestamp', 'max'),
    total_kL=('consumption_L', lambda x: f'{x.sum()/1000:.2f} kL'),
    debit_moyen=('flow_rate_Lmin', lambda x: f'{x.mean():.2f} L/min'),
).reset_index())

✅ Dataset unifié : 158,351 enregistrements
   Doublons supprimés, timestamps nettoyés, débit calculé



,customer,enregistrements,devices_uniques,debut,fin,total_kL,debit_moyen
0,A,42442,4,2025-05-31 09:34:15+00:00,2025-11-22 13:42:38+00:00,106.58 kL,4.79 L/min
1,B,15342,1,2025-10-06 11:08:13+00:00,2026-05-09 09:53:26+00:00,183.30 kL,12.25 L/min
2,C,73934,7,2024-03-05 14:57:20+00:00,2025-11-24 11:11:28+00:00,155.65 kL,3.18 L/min
3,Gym,26633,8,2024-04-22 11:38:13+00:00,2026-05-09 08:32:11+00:00,162.68 kL,4.89 L/min


---
## 3. 🔍 Couche Requêtes SQL-like (moteur pandas)

Fonction centrale de filtrage/agrégation. Toutes les fonctions analytiques l'utilisent.

In [30]:
def query_data(
    customers       = None,
    devices         = None,
    sub_categories  = None,
    tags            = None,      # 'hot' | 'cold'
    date_from       = None,
    date_to         = None,
    agg             = 'raw',     # 'raw' | 'daily' | 'hourly' | 'weekly' | 'monthly'
    metric          = 'consumption_L',
    top_n           = None,
) -> pd.DataFrame:
    """
    Moteur de requêtes central.
    Filtre, agrège et retourne un DataFrame selon les critères demandés.
    Supporte le filtrage multi-dimensions et les agrégations temporelles.
    """
    result = df.copy()

    if customers     : result = result[result['customer'].isin(customers)]
    if devices       : result = result[result['device'].isin(devices)]
    if sub_categories: result = result[result['sub_category_name'].isin(sub_categories)]
    if tags          : result = result[result['tag'].isin(tags)]
    if date_from     : result = result[result['timestamp'] >= pd.Timestamp(date_from, tz='UTC')]
    if date_to       : result = result[result['timestamp'] <= pd.Timestamp(date_to, tz='UTC')]

    if agg == 'daily':
        result = (result
            .groupby(['customer', 'device', 'date'], as_index=False)[metric]
            .agg(['sum', 'mean', 'count', 'max'])
            .rename(columns={'sum':'total', 'mean':'avg', 'count':'events', 'max':'peak'})
        )
    elif agg == 'hourly':
        result = (result
            .groupby(['customer', 'hour'], as_index=False)[metric].mean()
            .rename(columns={metric: 'avg_par_heure'})
        )
    elif agg == 'weekly':
        result = (result
            .groupby(['customer', 'week'], as_index=False)[metric].sum()
            .rename(columns={metric: 'total'})
        )
    elif agg == 'monthly':
        result = (result
            .groupby(['customer', 'month'], as_index=False)[metric].sum()
            .rename(columns={metric: 'total'})
        )

    if top_n:
        sort_col = result.columns[-1]
        result = result.nlargest(top_n, sort_col)

    return result.reset_index(drop=True)


# ── Test rapide ───────────────────────────────────────────────────────────────
test = query_data(customers=['C'], sub_categories=['Flush'], agg='daily')
print(f'✅ Test requête : Customer C / Flush / journalier → {test.shape}')
display(test.head(4))

✅ Test requête : Customer C / Flush / journalier → (1007, 7)


,customer,device,date,total,avg,events,peak
0,C,11d85810-daf7-11ee-b658-695385a245c5,2024-03-05,7.396,2.465333,3,4.718
1,C,11d85810-daf7-11ee-b658-695385a245c5,2024-03-06,142.627,7.923722,18,56.104
2,C,11d85810-daf7-11ee-b658-695385a245c5,2024-03-07,69.580,4.348750,16,6.839
3,C,11d85810-daf7-11ee-b658-695385a245c5,2024-03-08,65.883,4.705929,14,9.112


---
## 4. 🧠 Couche Sémantique — ChromaDB + Embeddings BGE

**Pourquoi ChromaDB ?**  
La recherche sémantique permet de retrouver des patterns similaires dans l'historique même si la question n'utilise pas les mêmes mots-clés que les données.

**BGE-small-en** : modèle d'embedding compact (33M paramètres), état de l'art sur les benchmarks MTEB.

In [31]:
import chromadb
from sentence_transformers import SentenceTransformer


def build_daily_documents(dataframe: pd.DataFrame):
    """
    Transforme chaque triplet (client, device, date) en document texte.
    Ces documents sont indexés dans ChromaDB pour la recherche sémantique.

    Format du document : descriptif riche incluant métriques clés
    pour permettre la correspondance sémantique avec les requêtes.
    """
    daily = dataframe.groupby(['customer', 'device', 'date']).agg(
        total_L       = ('consumption_L',   'sum'),
        events        = ('consumption_L',   'count'),
        peak_L        = ('consumption_L',   'max'),
        avg_flow      = ('flow_rate_Lmin',  'mean'),
        night_events  = ('is_night',        'sum'),
        is_weekend    = ('is_weekend',      'first'),
        sub_cat       = ('sub_category_name', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
        category      = ('main_category_name', 'first'),
        tag           = ('tag', lambda x: x.mode()[0] if not x.mode().empty else 'cold'),
    ).reset_index()

    docs, ids, metas = [], [], []
    for _, row in daily.iterrows():
        doc_id = f"{row['customer']}__{row['device'][:8]}__{row['date']}"

        # Document texte enrichi — optimisé pour la similarité sémantique
        day_type = 'weekend' if row['is_weekend'] else 'weekday'
        text = (
            f"Customer {row['customer']} | Site: {row['category']} | Usage: {row['sub_cat']} | "
            f"Water type: {row['tag']} | Date: {row['date']} ({day_type}) | "
            f"Total consumption: {row['total_L']:.2f} liters | "
            f"Usage events: {row['events']} | Peak single use: {row['peak_L']:.2f} L | "
            f"Average flow rate: {row['avg_flow']:.2f} L/min | "
            f"Nighttime events: {int(row['night_events'])} (potential leak indicator)"
        )
        meta = {
            'customer'    : row['customer'],
            'device'      : row['device'],
            'date'        : str(row['date']),
            'total_L'     : float(row['total_L']),
            'night_events': int(row['night_events']),
            'tag'         : row['tag'],
        }
        docs.append(text); ids.append(doc_id); metas.append(meta)

    return docs, ids, metas


print("🔄 Chargement du modèle d'embeddings BGE-small-en-v1.5...")
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5')

chroma_client   = chromadb.Client()
collection_name = 'watersec_daily_v2'
try:
    chroma_client.delete_collection(collection_name)
except:
    pass
collection = chroma_client.create_collection(collection_name)

print('📝 Génération des documents journaliers et indexation...')
docs, ids, metas = build_daily_documents(df)

# Indexation par batches (évite les timeouts sur gros volumes)
BATCH = 500
for i in range(0, len(docs), BATCH):
    batch_docs  = docs[i:i+BATCH]
    batch_ids   = ids[i:i+BATCH]
    batch_metas = metas[i:i+BATCH]
    embeddings  = embedder.encode(batch_docs, show_progress_bar=False).tolist()
    collection.add(documents=batch_docs, embeddings=embeddings,
                   ids=batch_ids, metadatas=batch_metas)
    print(f'  Indexé {min(i+BATCH, len(docs))}/{len(docs)} documents...', end='\r')

print(f'\n✅ ChromaDB : {len(docs):,} documents indexés')


def semantic_search(query: str, n_results: int = 5,
                    customer_filter: str = None) -> list:
    """Recherche sémantique dans la base vectorielle."""
    query_emb = embedder.encode([query]).tolist()
    where     = {'customer': customer_filter} if customer_filter else None
    results   = collection.query(
        query_embeddings=query_emb, n_results=n_results,
        where=where, include=['documents', 'metadatas', 'distances']
    )
    hits = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        hits.append({'document': doc, 'metadata': meta, 'score': 1 - dist})
    return hits


# Test
hits = semantic_search('high water consumption at night potential leak', n_results=3)
print(f'\n🔍 Test recherche sémantique — Top 3 résultats :')
for h in hits:
    print(f'  Score: {h["score"]:.3f} | {h["document"][:100]}...')

🔄 Chargement du modèle d'embeddings BGE-small-en-v1.5...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


📝 Génération des documents journaliers et indexation...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  Indexé 7723/7723 documents...
✅ ChromaDB : 7,723 documents indexés

🔍 Test recherche sémantique — Top 3 résultats :
  Score: 0.526 | Customer A | Site: Offices | Usage: unknown | Water type: cold | Date: 2025-08-14 (weekday) | Total ...
  Score: 0.515 | Customer A | Site: Offices | Usage: unknown | Water type: cold | Date: 2025-09-16 (weekday) | Total ...
  Score: 0.513 | Customer A | Site: Offices | Usage: unknown | Water type: cold | Date: 2025-09-20 (weekend) | Total ...


---
## 5. 🤖 Couche LLM — Groq API + Mémoire Conversationnelle

**Fonctionnalité** : L'agent maintient une mémoire des 5 derniers échanges (FIFO), permettant des questions de suivi comme *"Et pour le mois dernier ?"* ou *"Compare avec le client B"*.

In [34]:
from groq import Groq

# ── Prompt système v4 — expertise domaine eau enrichie ───────────────────────
WATER_SYSTEM_PROMPT =  """
Tu es WaterSec AI — agent expert en surveillance de consommation d'eau IoT.
Tu analyses des données de capteurs temps réel pour des clients professionnels et résidentiels.
 
━━━ RÈGLE N°1 : CONCISION (NON-NÉGOCIABLE) ━━━
Chaque réponse doit tenir en MAX 150 mots.
INTERDIT : introductions, reformulations, conclusions génériques, conseils non demandés.
AUTORISÉ  : chiffres, unités, noms de devices, actions concrètes.
Si tu veux écrire plus de 150 mots → coupe, ne justifie pas.
 
━━━ FORMAT SELON TYPE DE REQUÊTE ━━━
- Salutation          → 1 phrase max. Pas de données.
- Question factuelle  → 1 chiffre + 1 phrase. Fin.
- Analyse données     → exactement ce format, rien d'autre :
    📊 [conclusion en 1 phrase]
    🔢 [max 3 chiffres clés avec unités]
    ✅ [1 action concrète]
- Recommandations     → liste numérotée, max 5 items, 1 ligne chacun.
- Anomalie détectée   → device + valeur + seuil dépassé + action. Fin.
 
━━━ 🆕 FORMAT ANOMALIES SÉQUENTIELLES ━━━
Pour chaque anomalie séquentielle signalée, utilise TOUJOURS ce format :
  🕐 TEMPORAL_BREACH  → [device] actif à [heure]h — heure habituelle : [plage]. Suspect si hors profil > 3 fois.
  ⏱️ GAP_TOO_LONG     → [device] silencieux [N] min (attendu [M] min). Cause probable : [panne capteur | site vide | week-end].
  ⚡ GAP_TOO_SHORT    → [device] double-trigger en [N] sec. Cause probable : [fuite | erreur capteur].
  🔗 MISSING_SEQ      → [Flush] sans [Sink] dans [W] min ([N] fois ce mois). Risque : [hygiène | fuite non détectée].
 
━━━ 🆕 NIVEAUX DE PRIORITÉ DES ALERTES ━━━
Classe chaque anomalie avec un niveau avant de répondre :
 
🔴 CRITIQUE  (action immédiate < 1h)
  → Débit nocturne > 0.1 L/min pendant > 30 min (fuite active)
  → GAP_TOO_LONG > 48h sur device actif normalement
  → Consommation > 5× la médiane device sur une session
 
🟠 URGENT   (action < 24h)
  → TEMPORAL_BREACH répété > 3 fois le même device
  → GAP_TOO_SHORT répété (double-trigger probable = fuite intermittente)
  → Chasse d'eau > 15 L (Customer C) — défaut mécanisme
 
🟡 ATTENTION (surveillance renforcée)
  → MISSING_SEQ Flush→Sink > 20% des cas ce mois
  → Consommation > 2× médiane mais < 5× (pic inhabituel)
  → Ratio eau chaude Gym > 70% (thermostat à vérifier)
 
🟢 INFO      (pas d'action, juste noter)
  → Temporal breach isolé (1 occurrence)
  → Pic justifié météo ou prière (confiance < 0.5)
 
Signaler UNIQUEMENT les niveaux 🔴 et 🟠 comme alertes actives.
Les niveaux 🟡 et 🟢 → mentionner en fin de réponse sous "À surveiller".
 
━━━ LANGUE ━━━
Réponds TOUJOURS dans la langue de l'utilisateur (FR/AR/EN). Jamais de mélange.
 
━━━ BENCHMARKS EAU ━━━
Douche       : 50–80 L/usage        | Durée normale : 5–10 min
Chasse       : 3–6 L (éco) / 9–13 L | Chasse > 15 L = défaut mécanisme
Lavabo       : 0.5–2 L/usage
Wudu (وضوء)  : 1–2 L/ablution       | Durée : 1–3 min
Fuite active : débit nocturne > 0.1 L/min sur > 30 min consécutives
Anomalie     : > 3× médiane device  | Nuit (23h–6h) > 10% total → suspect
Gap normal entre usages (selon profil) :
  Gym douches   : 15–45 min entre sessions | Silence > 6h la nuit = normal
  Customer A    : silence week-end = normal | Silence Lun–Ven > 4h = suspect
  Customer B    : pics ±20 min prières = normaux
  Customer C    : silence 9h–17h semaine = normal (résidentiel)
 
━━━ CONTEXTE CLIENTS ━━━
A (Bureaux)    : actif Lun–Ven 7h–19h. Hors horaires = suspect.
                 Devices : blocs toilettes. Silence > 4h en heure de bureau → vérifier capteur.
B (Sanitaire)  : pics ±20 min des prières (Fajr/Dhuhr/Asr/Maghrib/Isha) = NORMAUX (ablution وضوء).
                 Chaque WC = 1 sink central + 2 chasses + 2 flexibles + 1 robinet وضوء.
C (Résidentiel): pics 6h–9h et 18h–22h normaux. Chasse > 15 L = défaut.
                 Sensors granulaires : sink + flush + flexible séparés → séquences attendues.
Gym (Douches)  : pics soir (18h–22h) + week-end normaux. Ratio chaud > 70% = thermostat.
                 4 cabines hot/cold séparées. Comparer cabines entre elles pour détecter dérive.
 
━━━ ANOMALIES CLASSIQUES : CHECKLIST AVANT ALERTE ━━━
1. T° > 28°C ?          → surconso froide attendue, pas une anomalie.
2. Heure de prière (B) ? → ablution normale, pas une anomalie.
3. Heure d'activité normale du profil ? → pic attendu, pas une anomalie.
4. Week-end / jour férié ? → silence attendu pour A, actif pour Gym.
Signaler UNIQUEMENT si confiance > 0.5 après ces 4 vérifications.
 
━━━ 🆕 ANOMALIES SÉQUENTIELLES : CHECKLIST ━━━
Pour TEMPORAL_BREACH :
  → Ce device a-t-il déjà été actif à cette heure au moins 3 fois ? → pas une anomalie.
  → Est-ce un jour de nettoyage/maintenance probable (lundi matin, vendredi soir) ? → signaler 🟡 seulement.
 
Pour GAP_TOO_LONG :
  → Est-ce un silence nocturne (23h–6h) ou de week-end pour ce profil ? → normal.
  → Le gap dépasse-t-il 48h sur un device normalement actif quotidiennement ? → 🔴 CRITIQUE.
 
Pour MISSING_SEQ (Flush sans Sink) :
  → Taux historique < 60% ? → la séquence n'est pas assez établie pour alerter.
  → Occurrence isolée ? → 🟢 INFO. Répétée > 5 fois ce mois ? → 🟠 URGENT.
 
━━━ 🆕 COMPARAISON INTER-DEVICES ━━━
Quand tu compares plusieurs devices du même type (ex: cabines Gym) :
  → Signale tout écart > 30% entre devices similaires comme anomalie relative.
  → Format : "Cabin 1 consomme X% de plus que la médiane des 4 cabines → vérifier [robinet | thermostat | capteur]"
 
━━━ RÉPONSE FINALE — STRUCTURE OBLIGATOIRE SI ANOMALIE ━━━
🚨 [Niveau : CRITIQUE | URGENT | ATTENTION]
📍 Device : [ID] | Client : [X] | Timestamp : [datetime]
📏 Valeur : [mesurée] vs [attendue/seuil]
🔎 Type : [TEMPORAL_BREACH | GAP_TOO_LONG | GAP_TOO_SHORT | MISSING_SEQ | VOLUME | DÉBIT]
💬 Explication : [1 phrase cause probable]
🔧 Action : [action concrète immédiate]
"""

# ── Mémoire conversationnelle (FIFO 5 échanges) ──────────────────────────────
conversation_memory = deque(maxlen=10)

# ── Mots-clés de salutation ───────────────────────────────────────────────────
GREETING_TOKENS = {
    'hi', 'hello', 'hey', 'bonjour', 'salut', 'salam', 'bonsoir',
    'ahlan', 'hola', 'ciao', 'مرحبا', 'السلام عليكم', 'اهلا',
    'howdy', 'greetings', 'sup', 'yo',
}

def _is_greeting(text: str) -> bool:
    tokens = set(text.lower().strip().rstrip('!,.').split())
    return bool(tokens & GREETING_TOKENS) or text.lower().strip() in GREETING_TOKENS


def call_llm(prompt: str, use_memory: bool = True, max_tokens: int = 600,
             system_override: str = None) -> str:
    if LLM_MODE == 'mock' or not GROQ_API_KEY:
        return f"[MODE MOCK] {prompt[:100]}...\n→ Configurer GROQ_API_KEY."

    # Budget tokens adaptatif selon type de message
    if _is_greeting(prompt):
        max_tokens = 60
    elif max_tokens == 600:
        # Limiter les réponses standard pour éviter le hors-sujet
        max_tokens = 350

    active_system = system_override if system_override else WATER_SYSTEM_PROMPT

    for attempt in range(2):
        try:
            client = Groq(api_key=GROQ_API_KEY)
            messages = [{"role": "system", "content": active_system}]
            if use_memory:
                messages.extend(list(conversation_memory))
            messages.append({"role": "user", "content": prompt})

            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                max_tokens=max_tokens,
                temperature=0.1,   # Très bas → factuel, pas de créativité hors-sujet
                top_p=0.85,
            )
            reply = response.choices[0].message.content

            conversation_memory.append({"role": "user",      "content": prompt})
            conversation_memory.append({"role": "assistant", "content": reply})
            return reply

        except Exception as e:
            err = str(e)
            if 'rate_limit' in err.lower() and attempt == 0:
                import time as _t
                print("⏳ Rate limit — retry dans 5s...")
                _t.sleep(5)
                continue
            return f"❌ Erreur LLM : {err}"
    return "❌ Erreur LLM persistante."


def reset_memory():
    conversation_memory.clear()
    print('🔄 Mémoire réinitialisée')


print('✅ LLM configuré | temperature=0.1 | max_tokens adaptatif (60/350/custom)')
print(f'   Mode : {LLM_MODE} | Modèle : {GROQ_MODEL}')

✅ LLM configuré | temperature=0.1 | max_tokens adaptatif (60/350/custom)
   Mode : groq | Modèle : llama-3.3-70b-versatile


---
## 5b. 🌦️🕌 Contexte Externe — Météo & Horaires de Prière

**Deux enrichissements contextuels innovants :**

| Module | API | Apport |
|--------|-----|--------|
| **Open-Meteo** | Gratuite, sans clé | Corrèle T° ambiante avec consommation eau |
| **Aladhan** | Gratuite, sans clé | Identifie les pics d'ablution (CustomerB) |

Ces modules permettent à l'agent de **distinguer les vraies anomalies** des variations attendues liées à la chaleur ou aux horaires de prière.


In [35]:
# ─── 🌦️ MODULE MÉTÉO — Open-Meteo (gratuit, sans clé API) ──────────────────
# Corrèle la température ambiante avec la consommation d'eau.
# Canicule (>35°C) → hausse attendue des douches froides (+25-40%)
# Gel (<5°C) → risque de fuites nocturnes sur tuyaux exposés
# ─────────────────────────────────────────────────────────────────────────────

WEATHER_CACHE = {}   # Cache par date pour limiter les appels API

def fetch_weather(date_str: str,
                  lat: float = SITE_LAT,
                  lon: float = SITE_LON) -> dict:
    """
    Récupère la météo historique via Open-Meteo Archive API (100 % gratuit).
    Retourne : temp_max, temp_min, temp_mean, precipitation, humidity.
    En cas d'échec réseau, retourne des valeurs None sans bloquer le pipeline.
    """
    if date_str in WEATHER_CACHE:
        return WEATHER_CACHE[date_str]
    try:
        url = (
            f"https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={lat}&longitude={lon}"
            f"&start_date={date_str}&end_date={date_str}"
            f"&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
            f"precipitation_sum,relative_humidity_2m_mean"
            f"&timezone=Africa/Tunis"
        )
        r = requests.get(url, timeout=6)
        d = r.json().get('daily', {})
        result = {
            'temp_max'     : d.get('temperature_2m_max',          [None])[0],
            'temp_min'     : d.get('temperature_2m_min',          [None])[0],
            'temp_mean'    : d.get('temperature_2m_mean',         [None])[0],
            'precipitation': d.get('precipitation_sum',           [None])[0],
            'humidity'     : d.get('relative_humidity_2m_mean',   [None])[0],
        }
        WEATHER_CACHE[date_str] = result
        return result
    except Exception:
        empty = {'temp_max': None, 'temp_min': None, 'temp_mean': None,
                 'precipitation': None, 'humidity': None}
        WEATHER_CACHE[date_str] = empty
        return empty


def get_weather_context_text(date_str: str) -> str:
    """Génère une ligne de contexte météo pour injection dans les prompts LLM."""
    w = fetch_weather(date_str)
    if w['temp_mean'] is None:
        return "Météo non disponible pour cette date."
    alert = ""
    if w['temp_max'] and w['temp_max'] > 35:
        alert = " ⚠️ CANICULE — surconsommation eau froide attendue (+25-40%)."
    elif w['temp_max'] and w['temp_max'] > 28:
        alert = " ☀️ Forte chaleur — pics douches froides probables."
    elif w['temp_min'] and w['temp_min'] < 5:
        alert = " 🧊 Risque gel — surveiller fuites nocturnes."
    rain = f" Précipitations: {w['precipitation']:.1f} mm." if w['precipitation'] is not None else ""
    return (
        f"Météo {date_str} : T° moy {w['temp_mean']:.1f}°C "
        f"(min {w['temp_min']:.1f} / max {w['temp_max']:.1f}).{rain}{alert}"
    )


# ─── 🕌 MODULE HORAIRES DE SALAT — Aladhan API (gratuit) ────────────────────
# CustomerB possède des robinets وضوء (ablution).
# Les pics de consommation aux heures de prière SONT NORMAUX.
# Ce module permet à l'agent de ne pas les signaler comme anomalies.
# ─────────────────────────────────────────────────────────────────────────────

PRAYER_CACHE = {}   # Cache par date

def fetch_prayer_times(date_str: str,
                       city    : str = SITE_CITY,
                       country : str = SITE_COUNTRY,
                       method  : int = PRAYER_METHOD) -> dict:
    """
    Récupère les horaires de prière via Aladhan API (gratuite).
    Retourne un dict : {Fajr, Dhuhr, Asr, Maghrib, Isha} au format "HH:MM".
    En cas d'échec, retourne des horaires approximatifs (Tunis, été).
    """
    if date_str in PRAYER_CACHE:
        return PRAYER_CACHE[date_str]
    try:
        url = (
            f"https://api.aladhan.com/v1/timingsByCity/{date_str}"
            f"?city={city}&country={country}&method={method}"
        )
        r       = requests.get(url, timeout=6)
        timings = r.json()['data']['timings']
        result  = {
            'Fajr'   : timings['Fajr'][:5],
            'Dhuhr'  : timings['Dhuhr'][:5],
            'Asr'    : timings['Asr'][:5],
            'Maghrib': timings['Maghrib'][:5],
            'Isha'   : timings['Isha'][:5],
        }
        PRAYER_CACHE[date_str] = result
        return result
    except Exception:
        # Horaires approximatifs Tunis, été (fallback sans connexion)
        fallback = {'Fajr': '04:30', 'Dhuhr': '12:30',
                    'Asr' : '16:00', 'Maghrib': '19:45', 'Isha': '21:00'}
        PRAYER_CACHE[date_str] = fallback
        return fallback


def is_prayer_time(timestamp: pd.Timestamp,
                   window_min: int = 20) -> tuple:
    """
    Vérifie si un timestamp tombe dans une fenêtre de ±window_min minutes
    autour d'un horaire de prière.
    Retourne (is_prayer: bool, prayer_name: str).
    Utile pour Customer B (robinets وضوء) — les pics sont normaux.
    """
    date_str   = str(timestamp.date())
    prayers    = fetch_prayer_times(date_str)
    t_min_val  = timestamp.hour * 60 + timestamp.minute
    for name, time_str in prayers.items():
        hh, mm       = map(int, time_str.split(':'))
        prayer_min   = hh * 60 + mm
        if abs(t_min_val - prayer_min) <= window_min:
            return True, name
    return False, ''


def get_prayer_context_text(date_str: str) -> str:
    """Résumé des horaires de prière pour injection dans les prompts LLM."""
    p = fetch_prayer_times(date_str)
    return (
        f"Horaires prière ({date_str}): "
        f"Fajr {p['Fajr']} | Dhuhr {p['Dhuhr']} | "
        f"Asr {p['Asr']} | Maghrib {p['Maghrib']} | Isha {p['Isha']}. "
        "Pics de consommation ±20 min de ces horaires = normaux (ablution ~1.5L)."
    )


print('✅ Module Météo (Open-Meteo) + Module Salat (Aladhan) initialisés')
print(f'   Site : {SITE_CITY}, {SITE_COUNTRY} | Méthode prière : {PRAYER_METHOD}')


✅ Module Météo (Open-Meteo) + Module Salat (Aladhan) initialisés
   Site : Tunis, Tunisia | Méthode prière : 3


---
## 6. 🔬 Couche ML — Détection d'Anomalies Multi-Modèle

**Deux approches complémentaires :**
1. **Isolation Forest** — détection non-supervisée des outliers multivariés
2. **🆕 Z-Score adaptatif** — détection statistique par fenêtre glissante (sensible aux dérives progressives)

In [36]:
def compute_anomaly_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Ingénierie de features pour la détection d'anomalies.
    Features : consommation, débit, heure, week-end, nuit,
               rolling stats (moyenne/écart-type 24h)
    """
    df_feat = dataframe.copy()

    # Features rolling par device (fenêtre 24 dernières mesures)
    df_feat = df_feat.sort_values(['device', 'timestamp'])
    df_feat['rolling_mean'] = (
        df_feat.groupby('device')['consumption_L']
        .transform(lambda x: x.rolling(24, min_periods=3).mean())
    )
    df_feat['rolling_std'] = (
        df_feat.groupby('device')['consumption_L']
        .transform(lambda x: x.rolling(24, min_periods=3).std().fillna(1))
    )
    df_feat['deviation_ratio'] = (
        (df_feat['consumption_L'] - df_feat['rolling_mean'])
        / df_feat['rolling_std'].replace(0, 1)
    ).fillna(0)

    return df_feat


def train_anomaly_models(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Entraîne un modèle Isolation Forest par client et un Z-Score adaptatif.
    Retourne le DataFrame annoté avec :
    - anomaly_label : 1 = anomalie, 0 = normal
    - anomaly_score : score de sévérité [0, 1]
    - anomaly_method : 'isolation_forest' | 'zscore'
    """
    FEATURE_COLS = ['consumption_L', 'flow_rate_Lmin', 'hour',
                    'is_weekend', 'is_night', 'deviation_ratio']

    df_ann = compute_anomaly_features(dataframe).copy()
    df_ann['anomaly_label']  = 0
    df_ann['anomaly_score']  = 0.0
    df_ann['anomaly_method'] = 'none'

    for customer in df_ann['customer'].unique():
        mask   = df_ann['customer'] == customer
        subset = df_ann[mask].copy()

        feat = subset[FEATURE_COLS].fillna(0)
        scaler = StandardScaler()
        X = scaler.fit_transform(feat)

        # ── Isolation Forest ─────────────────────────────────────────────
        iso = IsolationForest(
            contamination=0.03,   # 3% d'anomalies attendues
            n_estimators=200,     # Plus d'arbres = meilleure précision
            random_state=42
        )
        preds  = iso.fit_predict(X)
        scores = -iso.score_samples(X)  # Score positif = plus anormal

        # Normalisation du score [0, 1]
        s_min, s_max = scores.min(), scores.max()
        if s_max > s_min:
            scores_norm = (scores - s_min) / (s_max - s_min)
        else:
            scores_norm = scores * 0

        df_ann.loc[mask, 'anomaly_label']  = (preds == -1).astype(int)
        df_ann.loc[mask, 'anomaly_score']  = scores_norm
        df_ann.loc[mask, 'anomaly_method'] = 'isolation_forest'

        # ── 🆕 Z-Score adaptatif (détection des dérives progressives) ────
        z_scores = np.abs(subset['deviation_ratio'])
        zscore_anomalies = z_scores > 3.0  # Seuil : 3 sigma
        # Combiner avec Isolation Forest (OR logique)
        anomaly_idx = subset.index[zscore_anomalies]
        df_ann.loc[anomaly_idx, 'anomaly_label'] = 1
        df_ann.loc[anomaly_idx, 'anomaly_method'] = 'zscore+isoforest'
    # Statistiques
    n_anom = df_ann['anomaly_label'].sum()
    print(f'✅ Modèles ML entraînés | {n_anom:,} anomalies détectées '
          f'({100*n_anom/len(df_ann):.1f}% des données)')
    print(f'   Méthodes : Isolation Forest (200 arbres) + Z-Score adaptatif (σ=3)')

    return df_ann


df_ann = train_anomaly_models(df)

# Résumé par client
display(df_ann.groupby('customer').agg(
    total=('anomaly_label','count'),
    anomalies=('anomaly_label','sum'),
    pct=('anomaly_label', lambda x: f'{100*x.mean():.1f}%'),
    score_moyen=('anomaly_score', lambda x: f'{x[df_ann.loc[x.index,"anomaly_label"]==1].mean():.3f}'),
).reset_index())

# ─── 🆕 Enrichissement contextuel : météo + horaires de prière ───────────────
# Pour chaque anomalie, on calcule :
#   • weather_adjusted : la météo justifie-t-elle la consommation élevée ?
#   • is_prayer_time   : le timestamp coïncide-t-il avec un salat (CustomerB) ?
#   • prayer_name      : nom de la prière si applicable
#   • anomaly_confidence : score de confiance [0-1] corrigé par le contexte
#   • anomaly_reason   : explication textuelle lisible

print("\n🔄 Enrichissement contextuel météo + horaires de prière...")

weather_adjusted_lst = []
is_prayer_lst        = []
prayer_name_lst      = []
confidence_lst       = []
reason_lst           = []

for _, row in df_ann.iterrows():
    ts = row['timestamp']

    # — Contexte météo ─────────────────────────────────────────────────────
    temp       = row.get('temp_mean')   # None si non enrichi
    heat_adj   = False
    heat_derate = 0.0
    if temp is not None:
        if temp > 35:
            heat_adj    = True
            heat_derate = 0.40
        elif temp > 28:
            heat_adj    = True
            heat_derate = 0.20

    # — Contexte prière (seulement CustomerB — wudu سinks) ─────────────────
    is_prayer, p_name = False, ''
    if row['customer'] == 'B':
        is_prayer, p_name = is_prayer_time(ts, window_min=20)

    # — Score de confiance corrigé ─────────────────────────────────────────
    base_conf = float(row['anomaly_score'])   # déjà normalisé [0,1]
    confidence = base_conf
    if row['anomaly_label'] == 1:
        if is_prayer:
            confidence *= 0.25   # très probablement ablution → quasi faux-positif
        if heat_adj:
            confidence *= (1.0 - heat_derate)

    # — Raison textuelle ───────────────────────────────────────────────────
    if row['anomaly_label'] == 0:
        reason = 'Normal'
    elif is_prayer:
        reason = f'Probable ablution (salat {p_name}) — faux-positif probable'
    elif row.get('is_night', 0) and row.get('flow_rate_Lmin') and row['flow_rate_Lmin'] > 0.1:
        reason = f'Débit nocturne suspect → fuite potentielle ({row["flow_rate_Lmin"]:.2f} L/min)'
    elif heat_adj:
        reason = f'Volume élevé justifié par la chaleur (T° moy {temp:.1f}°C)'
    elif row['consumption_L'] > 100:
        reason = f'Volume anormalement élevé en une session ({row["consumption_L"]:.1f} L)'
    else:
        reason = f'Pattern inhabituel (score ML : {base_conf:.2f})'

    weather_adjusted_lst.append(heat_adj)
    is_prayer_lst.append(is_prayer)
    prayer_name_lst.append(p_name)
    confidence_lst.append(round(min(confidence, 1.0), 3))
    reason_lst.append(reason)

df_ann['weather_adjusted']  = weather_adjusted_lst
df_ann['is_prayer_time']    = is_prayer_lst
df_ann['prayer_name']       = prayer_name_lst
df_ann['anomaly_confidence'] = confidence_lst
df_ann['anomaly_reason']    = reason_lst

n_true   = ((df_ann['anomaly_label'] == 1) & (df_ann['anomaly_confidence'] > 0.5)).sum()
n_prayer = df_ann['is_prayer_time'].sum()
n_heat   = df_ann['weather_adjusted'].sum()
print(f'✅ Enrichissement terminé :')
print(f'   • Anomalies haute confiance (>0.5) : {n_true}')
print(f'   • Événements aux heures de prière  : {n_prayer}')
print(f'   • Événements ajustés par la météo  : {n_heat}')

✅ Modèles ML entraînés | 6,548 anomalies détectées (4.1% des données)
   Méthodes : Isolation Forest (200 arbres) + Z-Score adaptatif (σ=3)


,customer,total,anomalies,pct,score_moyen
0,A,42442,1566,3.7%,0.640
1,B,15342,654,4.3%,0.658
2,C,73934,3213,4.3%,0.608
3,Gym,26633,1115,4.2%,0.644



🔄 Enrichissement contextuel météo + horaires de prière...
✅ Enrichissement terminé :
   • Anomalies haute confiance (>0.5) : 5296
   • Événements aux heures de prière  : 3000
   • Événements ajustés par la météo  : 0


---
## 7. Water Demand Index (WDI) & Score d'Efficacité

**Water Demand Index (WDI)** : Courbe de demande journalière normalisée permettant de comparer les profils d'usage entre clients différents, indépendamment du volume absolu.

**Score d'efficacité hydrique** : KPI composite [0-100] mesurant l'efficacité par rapport aux benchmarks sectoriels.

In [37]:
# ── Water Demand Index (WDI) ──────────────────────────────────────────────────
def compute_wdi(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Calcule le Water Demand Index par client et par heure.
    WDI = consommation horaire / consommation horaire maximale
    → Valeur entre 0 et 1, comparable entre clients de tailles différentes.
    """
    hourly = (dataframe
        .groupby(['customer', 'hour'])['consumption_L']
        .mean()
        .reset_index()
        .rename(columns={'consumption_L': 'avg_L'})
    )
    # Normalisation par client
    max_by_customer = hourly.groupby('customer')['avg_L'].max()
    hourly['WDI'] = hourly.apply(
        lambda r: r['avg_L'] / max_by_customer[r['customer']]
        if max_by_customer[r['customer']] > 0 else 0, axis=1
    )
    return hourly


wdi = compute_wdi(df)

fig_wdi = px.line(
    wdi, x='hour', y='WDI', color='customer',
    color_discrete_map=COLORS,
    title='💧 Water Demand Index (WDI) — Profil de demande normalisé par client',
    labels={'hour': 'Heure de la journée', 'WDI': 'Water Demand Index (0-1)'},
    template='plotly_white'
)
fig_wdi.update_xaxes(dtick=1)
fig_wdi.update_layout(
    annotations=[dict(
        x=12, y=0.05, text='WDI = 1.0 : pic de consommation | WDI = 0.0 : pas de consommation',
        showarrow=False, font=dict(size=10, color='gray')
    )]
)
fig_wdi.show()


# ── Score d'efficacité hydrique ───────────────────────────────────────────────
def compute_water_efficiency_score(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Score composite d'efficacité hydrique [0-100] par client.

    Composantes (pondération) :
    - Volume nocturne %      (30%) → faible = efficient
    - Débit moyen vs benchmark (25%) → proche benchmark = efficient
    - Variabilité            (25%) → faible CV = patterns réguliers
    - Taux d'anomalies       (20%) → faible = efficient
    """
    scores = []

    for customer in dataframe['customer'].unique():
        sub = dataframe[dataframe['customer'] == customer]

        # 1. Score nocturne (30%) — plus le % nocturne est bas, mieux c'est
        night_pct  = sub['is_night'].mean()
        score_night = max(0, 1 - night_pct * 5)  # Pénalité si > 20% nocturne

        # 2. Score de débit (25%) — débit proche du benchmark = efficient
        avg_flow    = sub['flow_rate_Lmin'].median()
        bench_flow  = 3.0  # Benchmark général : 3 L/min
        score_flow  = max(0, 1 - abs(avg_flow - bench_flow) / bench_flow)

        # 3. Score de régularité (25%) — coefficient de variation
        daily_totals = sub.groupby('date')['consumption_L'].sum()
        if daily_totals.std() > 0:
            cv = daily_totals.std() / daily_totals.mean()
            score_regularity = max(0, 1 - cv / 2)
        else:
            score_regularity = 1.0

        # 4. Score anomalies (20%)
        if 'anomaly_label' in sub.columns:
            anom_rate   = sub['anomaly_label'].mean()
            score_anom  = max(0, 1 - anom_rate * 10)
        else:
            score_anom = 0.5

        # Score composite pondéré
        composite = (0.30 * score_night + 0.25 * score_flow +
                     0.25 * score_regularity + 0.20 * score_anom) * 100

        scores.append({
            'client'          : customer,
            'score_nocturne'  : round(score_night * 100, 1),
            'score_debit'     : round(score_flow * 100, 1),
            'score_regularite': round(score_regularity * 100, 1),
            'score_anomalies' : round(score_anom * 100, 1),
            'SCORE_GLOBAL'    : round(composite, 1),
        })

    return pd.DataFrame(scores).sort_values('SCORE_GLOBAL', ascending=False)


df_scores = compute_water_efficiency_score(df_ann)
print("\n📊 Score d'efficacité hydrique par client :")
display(df_scores)

fig_score = px.bar(
    df_scores.melt(id_vars='client', var_name='Composante', value_name='Score'),
    x='client', y='Score', color='Composante', barmode='group',
    title="🏆 Score d'efficacité hydrique par client (0-100)",
    template='plotly_white'
)
fig_score.show()


📊 Score d'efficacité hydrique par client :


,client,score_nocturne,score_debit,score_regularite,score_anomalies,SCORE_GLOBAL
0,C,98.0,99.6,70.4,56.5,83.2
1,A,97.9,80.0,65.5,63.1,78.4
2,Gym,90.0,67.7,67.6,58.1,72.5
3,B,57.9,0.0,66.9,57.4,45.6


---
## 8. Détection de Fuites Continues (Nuit)

**Algorithme** : Une fuite est définie comme un débit persistant nocturne (23h-6h) supérieur au seuil de 0.1 L/min pendant au moins 30 minutes consécutives.

In [38]:
def detect_continuous_leaks(
    dataframe     : pd.DataFrame,
    flow_threshold: float = 0.1,   # L/min — seuil de fuite
    min_duration  : int   = 30,    # minutes — durée minimale
) -> pd.DataFrame:
    """
    Détecte les séquences de débit nocturne persistant.

    Méthode :
    1. Filtrage des événements nocturnes avec débit > seuil
    2. Regroupement en séquences continues (gap < 10 min)
    3. Filtrage par durée minimale
    4. Calcul du volume perdu estimé
    """
    night = dataframe[
        (dataframe['is_night'] == 1) &
        (dataframe['flow_rate_Lmin'] > flow_threshold)
    ].copy().sort_values(['device', 'timestamp'])

    if night.empty:
        return pd.DataFrame()

    leaks = []

    for device, group in night.groupby('device'):
        group = group.sort_values('timestamp').reset_index(drop=True)
        customer = group['customer'].iloc[0]

        # Détection des séquences continues (gap < 10 min entre événements)
        group['time_gap'] = group['timestamp'].diff().dt.total_seconds().fillna(0) / 60
        group['seq_id']   = (group['time_gap'] > 10).cumsum()

        for seq_id, seq in group.groupby('seq_id'):
            if len(seq) < 2:
                continue
            duration_min = (seq['timestamp'].max() - seq['timestamp'].min()).total_seconds() / 60

            if duration_min >= min_duration:
                volume_lost = seq['consumption_L'].sum()
                avg_flow    = seq['flow_rate_Lmin'].mean()
                leaks.append({
                    'customer'    : customer,
                    'device'      : device,
                    'start_time'  : seq['timestamp'].min(),
                    'end_time'    : seq['timestamp'].max(),
                    'duration_min': round(duration_min, 1),
                    'volume_L'    : round(volume_lost, 2),
                    'avg_flow_Lmin': round(avg_flow, 3),
                    'events'      : len(seq),
                    'severity'    : 'HIGH' if volume_lost > 10 else 'MEDIUM' if volume_lost > 3 else 'LOW',
                })

    return pd.DataFrame(leaks).sort_values('volume_L', ascending=False).reset_index(drop=True)


df_leaks = detect_continuous_leaks(df_ann, flow_threshold=0.1, min_duration=30)

if not df_leaks.empty:
    print(f'🚨 {len(df_leaks)} séquences de fuites nocturnes détectées')
    print(f'   Volume total potentiellement perdu : {df_leaks["volume_L"].sum():.1f} L')
    display(df_leaks.head(10))

    # Visualisation des fuites par client et sévérité
    if 'severity' in df_leaks.columns:
        fig_leaks = px.scatter(
            df_leaks,
            x='start_time', y='volume_L',
            color='customer', size='duration_min',
            symbol='severity',
            color_discrete_map=COLORS,
            title='🚨 Fuites nocturnes détectées (taille = durée)',
            labels={'volume_L': 'Volume perdu (L)', 'start_time': 'Date/heure début'},
            hover_data=['duration_min', 'avg_flow_Lmin', 'device'],
            template='plotly_white'
        )
        fig_leaks.show()

    # Demande LLM d'interprétation
    top3 = df_leaks.head(3)[['customer','start_time','duration_min','volume_L','severity']].to_string(index=False)
    print('\n💬 Analyse LLM des fuites :')
    print(call_llm(
        f"I detected {len(df_leaks)} nighttime leak sequences. Top 3:\n{top3}\n"
        "Provide a brief risk assessment and maintenance action plan."
    ))
else:
    print("✅ Aucune fuite nocturne persistante détectée avec les seuils actuels")

🚨 6 séquences de fuites nocturnes détectées
   Volume total potentiellement perdu : 1559.7 L


,customer,device,start_time,end_time,duration_min,volume_L,avg_flow_Lmin,events,severity
0,Gym,8161ea40-4a9c-11ef-82d3-2ffa8384e699,2026-01-31 03:23:26+00:00,2026-01-31 05:50:08+00:00,146.7,590.82,11.849,168,HIGH
1,Gym,8161ea40-4a9c-11ef-82d3-2ffa8384e699,2026-01-30 23:01:17+00:00,2026-01-31 00:32:10+00:00,90.9,331.00,16.307,88,HIGH
2,B,da76d010-9f81-11f0-ba67-d14dea579ee6,2025-11-19 04:33:48+00:00,2025-11-19 05:30:08+00:00,56.3,254.58,14.267,23,HIGH
3,B,da76d010-9f81-11f0-ba67-d14dea579ee6,2025-11-20 04:50:47+00:00,2025-11-20 05:23:07+00:00,32.3,188.91,12.861,11,HIGH
4,Gym,8161ea40-4a9c-11ef-82d3-2ffa8384e699,2026-01-31 02:31:24+00:00,2026-01-31 03:13:20+00:00,41.9,185.36,14.093,29,HIGH
5,A,37de7e10-3df4-11f0-bf6d-efc7d3c1cde5,2025-08-14 04:33:41+00:00,2025-08-14 05:58:46+00:00,85.1,9.02,0.524,54,MEDIUM



💬 Analyse LLM des fuites :
🚨 Niveau : 🔴 CRITIQUE
📍 Device : Gym | Client : Gym | Timestamp : 2026-01-31 03:23:26+00:00
📏 Valeur : 590.82 L vs seuil
🔎 Type : Fuite active
💬 Explication : Fuite nocturne importante au Gym.
🔧 Action : Vérifier les installations d'eau du Gym immédiatement, notamment les cabines de douche et les robinets. 
À surveiller : Customer B pour fuite potentielle.


---
## 9. Analyse Comportementale Séquentielle (Chaînes de Markov)

**Fonctionnalité** : Modélisation des transitions entre types d'usage (Flush → Sink → Tap) comme une chaîne de Markov. Permet de détecter des comportements attendus vs inattendus et de comprendre les habitudes d'usage.

In [39]:
def detect_sequences(customer='C', event_a='Flush', event_b='Sink', max_gap_min=5):
    """
    Détecte les co-occurrences temporelles entre deux types d'événements.
    Retourne les paires (event_a → event_b) dans la fenêtre de temps donnée.
    """
    df_cust = df[df['customer'] == customer].copy()
    ea = df_cust[df_cust['sub_category_name'] == event_a][['timestamp','consumption_L']].sort_values('timestamp')
    eb = df_cust[df_cust['sub_category_name'] == event_b][['timestamp','consumption_L']].sort_values('timestamp')

    ea = ea.rename(columns={'timestamp': 'ts_a', 'consumption_L': 'conso_a'})
    eb = eb.rename(columns={'timestamp': 'ts_b', 'consumption_L': 'conso_b'})

    merged = pd.merge_asof(
        ea, eb,
        left_on='ts_a', right_on='ts_b',
        direction='forward',
        tolerance=pd.Timedelta(f'{max_gap_min}min')
    )
    seqs = merged[merged['ts_b'].notna()].copy()
    seqs['gap_seconds'] = (seqs['ts_b'] - seqs['ts_a']).dt.total_seconds()
    seqs['hour_a']      = seqs['ts_a'].dt.hour
    return seqs


def build_markov_matrix(customer='C', max_gap_min=5) -> pd.DataFrame:
    """
    Construit la matrice de transition de Markov entre types d'usage.
    P(B|A) = probabilité que B survienne dans les max_gap_min minutes après A.
    """
    sub_cats = df[df['customer'] == customer]['sub_category_name'].dropna().unique().tolist()
    sub_cats = [s for s in sub_cats if s]  # Supprime None/NaN

    matrix = {}
    for cat_a in sub_cats:
        total_a = len(df[(df['customer'] == customer) & (df['sub_category_name'] == cat_a)])
        if total_a == 0:
            continue
        row = {}
        for cat_b in sub_cats:
            seqs    = detect_sequences(customer, cat_a, cat_b, max_gap_min)
            prob    = len(seqs) / total_a
            row[cat_b] = round(prob, 3)
        matrix[cat_a] = row

    return pd.DataFrame(matrix).T.fillna(0)


# ── Customer C : matrice de Markov ───────────────────────────────────────────
print('🔄 Construction de la matrice de Markov pour Customer C...')
markov = build_markov_matrix('C', max_gap_min=5)

if not markov.empty:
    print('\n📊 Matrice de transitions (P(B|A) dans les 5 min) :')
    display(markov)

    fig_markov = px.imshow(
        markov,
        color_continuous_scale='Blues',
        title='🔗 Chaîne de Markov — Probabilités de transition entre usages (Customer C)',
        labels=dict(x='Événement suivant (B)', y='Événement précédent (A)', color='P(B|A)'),
        text_auto='.3f',
        template='plotly_white'
    )
    fig_markov.show()

    # Séquence Flush → Sink détaillée
    seqs_fs = detect_sequences('C', 'Flush', 'Sink', 5)
    pct_fs  = 100 * len(seqs_fs) / max(1, len(df[(df['customer']=='C') & (df['sub_category_name']=='Flush')]))
    print(f'\n📈 Séquences Flush → Sink (< 5 min) : {len(seqs_fs):,} ({pct_fs:.1f}% des chasses)')

    # Distribution par heure
    fig_seq = px.histogram(
        seqs_fs, x='hour_a', nbins=24,
        title='Customer C — Distribution horaire des séquences Flush → Sink',
        labels={'hour_a': 'Heure', 'count': 'Nombre de séquences'},
        color_discrete_sequence=['#FF5722'],
        template='plotly_white'
    )
    fig_seq.update_xaxes(dtick=1)
    fig_seq.show()

    # Interprétation LLM
    markov_str   = markov.to_string()
    seq_summary  = seqs_fs.groupby('hour_a')['gap_seconds'].agg(['count','mean']).round(1).to_string()
    print('\n💬 Interprétation LLM des patterns comportementaux :')
    print(call_llm(
        f"Markov transition matrix for Customer C (residential bathroom):\n{markov_str}\n\n"
        f"Flush→Sink sequences by hour (count, avg gap in seconds):\n{seq_summary}\n\n"
        "Interpret the behavioral patterns and their water efficiency implications."
    ))

🔄 Construction de la matrice de Markov pour Customer C...

📊 Matrice de transitions (P(B|A) dans les 5 min) :


,Sink,Flush,Tap
Sink,1.000,0.613,0.303
Flush,0.515,1.000,0.349
Tap,0.817,0.950,1.000



📈 Séquences Flush → Sink (< 5 min) : 11,878 (51.5% des chasses)



💬 Interprétation LLM des patterns comportementaux :
📊 Les habitudes d'utilisation de l'eau à la résidence de Customer C suivent un modèle prévisible.
🔢 1562 séquences Flush→Sink à 11h, 110,7 secondes de gap moyen.
✅ Vérifier les installations pour optimiser la consommation d'eau, notamment pendant les heures de pointe (7h-12h). 
À surveiller : Utilisation de l'eau en soirée (18h-22h) pour détecter d'éventuelles fuites ou gaspillages.


---
## 10. Prévision Prophet 7 jours

**Prophet** (Meta) : modèle de prévision de séries temporelles robuste aux données manquantes et aux tendances non-linéaires. Idéal pour les données de consommation d'eau avec saisonnalité hebdomadaire.

In [40]:
# Cellule de séparation — espace réservé pour tests intermédiaires
pass

In [41]:
# Cellule de séparation — espace réservé pour tests intermédiaires
pass

In [42]:
from prophet import Prophet
print("✅ Prophet importé avec succès")

✅ Prophet importé avec succès


In [ ]:
def forecast_consumption(customer: str, days: int = 7):
    """
    Prédit la consommation d'eau journalière pour un client donné sur les `days` suivants
    en utilisant le modèle Prophet.
    Retourne : DataFrame de prévision, Plotly Figure.
    """
    cust_df = df[df['customer'] == customer].copy()
    if cust_df.empty:
        print(f"Pas de données pour le client {customer}.")
        return None, None

    # Agrégation journalière pour Prophet
    daily_consumption = cust_df.groupby('date')['consumption_L'].sum().reset_index()
    daily_consumption['date'] = pd.to_datetime(daily_consumption['date'])

    # Prophet nécessite des colonnes 'ds' (datestamp) et 'y' (valeur à prédire)
    prophet_df = daily_consumption.rename(columns={'date': 'ds', 'consumption_L': 'y'})

    if len(prophet_df) < 20: # Prophet a besoin d'un minimum de données historiques
        print(f"Pas assez de données historiques pour le client {customer} pour la prévision.")
        return None, None

    # Initialisation et entraînement du modèle Prophet
    model = Prophet(
        seasonality_mode='multiplicative',
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=False,
        changepoint_prior_scale=0.05
    )
    model.fit(prophet_df)

    # Création d'un DataFrame pour les prédictions futures
    future = model.make_future_dataframe(periods=days, include_history=True, freq='D')

    # Prédiction
    forecast = model.predict(future)

    # Visualisation
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=prophet_df['ds'], y=prophet_df['y'], mode='markers', name='Historique',
                             marker=dict(color=COLORS.get(customer, '#636EFA'))))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], mode='lines', name='Prédiction',
                             line=dict(color='orange')))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], mode='lines', name='Intervalle inf.',
                             line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], mode='lines', name='Intervalle sup.',
                             fill='tonexty', fillcolor='rgba(255,165,0,0.2)', line=dict(width=0)))

    fig.update_layout(
        title=f'Prévision de Consommation pour le Client {customer}',
        xaxis_title='Date',
        yaxis_title='Consommation (L)',
        template='plotly_white'
    )

    return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], fig

print('✅ Fonction `forecast_consumption` définie.')

---
## 11. 🏋️ Dashboard Gym — Comparaison des Cabines

Analyse détaillée des 4 cabines douche (hot + cold) avec benchmarking inter-cabines.

In [44]:
df_gym      = df[df['customer'] == 'Gym'].copy()
device_list = sorted(df_gym['device'].unique())
cabin_map   = {d: f'Cabin {i+1}' for i, d in enumerate(device_list)}
df_gym['cabin'] = df_gym['device'].map(cabin_map)

# ── Consommation journalière par cabine ──────────────────────────────────────
daily_gym = (df_gym
    .groupby(['cabin', 'date', 'tag'])
    .agg(total_L=('consumption_L','sum'), events=('consumption_L','count'))
    .reset_index()
)
daily_gym['date'] = pd.to_datetime(daily_gym['date'])

# ── Graphique 1 : Tendance journalière ──────────────────────────────────────
fig_gym1 = px.line(
    daily_gym[daily_gym['tag'] == 'hot'],
    x='date', y='total_L', color='cabin',
    title='🏋️ Gym — Consommation EAU CHAUDE par cabine (L/jour)',
    labels={'total_L': 'L/jour'},
    template='plotly_white'
)


# ── Graphique 2 : Ratio chaud/froid par cabine ───────────────────────────────
pivot_hc = (daily_gym
    .groupby(['cabin', 'tag'])['total_L'].mean()
    .reset_index()
    .pivot(index='cabin', columns='tag', values='total_L')
    .fillna(0)
)
if 'hot' in pivot_hc.columns and 'cold' in pivot_hc.columns:
    pivot_hc['hot_ratio'] = pivot_hc['hot'] / (pivot_hc['hot'] + pivot_hc['cold'] + 1e-9) * 100

    fig_ratio = px.bar(
        pivot_hc.reset_index(),
        x='cabin', y=['hot', 'cold'],
        title='💧🔥 Gym — Ratio eau chaude/froide par cabine (L/jour moyen)',
        labels={'value': 'L/jour', 'variable': 'Type'},
        color_discrete_map={'hot': '#FF5722', 'cold': '#2196F3'},
        template='plotly_white'
    )
    
    display(pivot_hc.round(2))

# ── Statistiques et outliers ─────────────────────────────────────────────────
cabin_stats    = daily_gym.groupby('cabin')['total_L'].agg(['mean','median','std','max']).round(1)
overall_median = cabin_stats['median'].median()
outlier_cabins = cabin_stats[cabin_stats['median'] > 1.5 * overall_median].index.tolist()

print('\n📊 Statistiques par cabine (L/jour) :')
display(cabin_stats)
print(f'Benchmark médian général : {overall_median:.1f} L/jour')
print(f'Cabines hors-norme (>1.5× benchmark) : {outlier_cabins if outlier_cabins else "Aucune"}')

print('\n💬 Analyse LLM :')
print(call_llm(
    f"Gym shower cabin statistics (L/day):\n{cabin_stats.to_string()}\n"
    f"Benchmark: {overall_median:.1f} L/day | Outliers: {outlier_cabins}\n"
    "Explain operationally and suggest maintenance priorities."
))


📊 Statistiques par cabine (L/jour) :


,mean,median,std,max
cabin,,,,
Cabin 1,38.3,15.0,122.8,1633.7
Cabin 2,62.2,48.1,54.6,310.9
Cabin 3,13.5,5.2,18.4,118.5
Cabin 4,18.1,13.1,19.2,109.1
Cabin 5,24.8,11.3,30.8,153.5
Cabin 6,91.3,70.4,75.7,428.0
Cabin 7,79.1,72.9,52.3,338.7
Cabin 8,16.5,10.4,19.0,101.7


Benchmark médian général : 14.1 L/jour
Cabines hors-norme (>1.5× benchmark) : ['Cabin 2', 'Cabin 6', 'Cabin 7']

💬 Analyse LLM :
📊 Les cabines de douche du Gym présentent des disparités de consommation d'eau.
🔢 Cabin 6 : 91,3 L/jour, Cabin 2 : 62,2 L/jour, Cabin 7 : 79,1 L/jour.
✅ Vérifier les robinets et les têtes de douche des cabines 2, 6 et 7 pour détecter d'éventuelles fuites ou réglages incorrects. 
À surveiller : Cabin 1 pour fuite potentielle en raison de la valeur maximale élevée (1633,7 L/jour).


---
## 12. 🌙 Dashboard Risque de Fuites Nocturnes

Matrice de risque (device × mois) des anomalies nocturnes — outil d'aide à la maintenance préventive.

In [45]:
def night_leak_dashboard(threshold_score: float = 0.45):
    """
    Construit une matrice de risque de fuites nocturnes.
    Axe X : mois, Axe Y : device, Valeur : nombre d'anomalies nocturnes.
    """
    night_anom = df_ann[
        (df_ann['is_night']       == 1) &
        (df_ann['anomaly_label']  == 1) &
        (df_ann['anomaly_score']  >= threshold_score)
    ].copy()

    if night_anom.empty:
        print('✅ Aucune anomalie nocturne au-dessus du seuil')
        return

    night_anom['month_label']  = night_anom['timestamp'].dt.strftime('%Y-%m')
    night_anom['device_short'] = night_anom['customer'] + ' | ' + night_anom['device'].str[:6]

    pivot = (night_anom
        .groupby(['device_short', 'month_label'])
        .size()
        .reset_index(name='count')
        .pivot(index='device_short', columns='month_label', values='count')
        .fillna(0)
    )

    fig = px.imshow(
        pivot,
        color_continuous_scale='Reds',
        title=f'🌙 Matrice de Risque Fuites Nocturnes — Anomalies/device/mois (score ≥ {threshold_score})',
        labels=dict(x='Mois', y='Device', color='Nb anomalies'),
        aspect='auto',
        template='plotly_white'
    )
    fig.update_layout(height=max(300, len(pivot) * 25))
    fig.show()

    # Top 10 devices à risque
    top_risk = (night_anom
        .groupby(['customer', 'device'])
        .agg(
            anomalies_nocturnes=('anomaly_label',  'count'),
            score_moyen        =('anomaly_score',  'mean'),
            debit_moyen_Lmin   =('flow_rate_Lmin', 'mean'),
            volume_total_L     =('consumption_L',  'sum'),
        )
        .reset_index()
        .sort_values('anomalies_nocturnes', ascending=False)
        .head(10)
        .round(3)
    )
    print('🔴 Top 10 devices à risque de fuite :')
    display(top_risk)

    risk_str = top_risk.to_string(index=False)
    print("\n💬 Plan d'action LLM :")
    print(call_llm(
        f"Night leak risk analysis — Top 10 high-risk devices:\n{risk_str}\n"
        "Provide a prioritized maintenance action plan with urgency levels."
    ))
    return top_risk


risk_table = night_leak_dashboard(threshold_score=0.45)

🔴 Top 10 devices à risque de fuite :


,customer,device,anomalies_nocturnes,score_moyen,debit_moyen_Lmin,volume_total_L
3,B,da76d010-9f81-11f0-ba67-d14dea579ee6,291,0.715,16.929,8614.155
11,Gym,8161ea40-4a9c-11ef-82d3-2ffa8384e699,291,0.762,13.420,1131.441
1,A,37de7e10-3df4-11f0-bf6d-efc7d3c1cde5,138,0.719,0.630,46.847
4,C,11d7eb90-daf7-11ee-a649-f9227cd29f6a,120,0.694,2.040,69.157
16,Gym,81689b30-4a9c-11ef-ad50-1d2daba32da0,54,0.692,5.017,877.292
17,Gym,816932d0-4a9c-11ef-b3c5-85b5ce342d21,51,0.715,4.794,981.453
10,C,11da4960-daf7-11ee-a71f-9b7fa292542a,41,0.716,1.012,9.273
6,C,11d8a970-daf7-11ee-b288-f5d6bb92dfd4,38,0.695,2.913,29.005
5,C,11d85810-daf7-11ee-b658-695385a245c5,31,0.709,5.156,147.843
9,C,11d9e340-daf7-11ee-a3c3-f51ff845e206,31,0.696,2.633,27.968



💬 Plan d'action LLM :
🚨 Niveau : 🔴 CRITIQUE
📍 Device : da76d010-9f81-11f0-ba67-d14dea579ee6 | Customer : B
📏 Valeur : 8614,155 L vs seuil
🔎 Type : Fuite nocturne
💬 Explication : Fuite nocturne importante chez Customer B.
🔧 Action : Vérifier immédiatement les installations d'eau de Customer B.
À surveiller : Les devices des Customers A, C et Gym pour fuites potentielles.


---
## 13. 🤖 Orchestrateur LangGraph — Agent Conversationnel

L'agent utilise un routeur LLM (plus intelligent que les regex) pour identifier l'intention, puis délègue au nœud spécialisé approprié.

In [46]:
from langgraph.graph import StateGraph, END


# ── État de l'agent ──────────────────────────────────────────────────────────
class AgentState(TypedDict):
    user_query     : str
    intent         : str
    data_context   : Optional[Any]
    anomaly_context: Optional[Any]
    semantic_hits  : Optional[List]
    chart          : Optional[Any]
    llm_response   : str
    final_answer   : str
    context_info   : str   # 🆕 Contexte météo + prières injecté dans les prompts


# ── Routeur — classification d'intention ─────────────────────────────────────
# ── 🆕 Mots-clés de salutation (priorité absolue dans le routeur) ────────────
GREETING_WORDS = {
    'hi', 'hello', 'hey', 'bonjour', 'salut', 'salam', 'bonsoir',
    'ahlan', 'hola', 'ciao', 'مرحبا', 'السلام عليكم', 'اهلا',
}

INTENT_KEYWORDS = {
    'query'         : ['average', 'total', 'how much', 'consumption', 'table',
                       'compare', 'between', 'show me', 'what is', 'list',
                       'moyenne', 'total', 'combien', 'tableau'],
    'analysis'      : ['trend', 'plot', 'chart', 'graph', 'daily', 'weekly',
                       'monthly', 'profile', 'compare', 'visualize', 'distribution',
                       'tendance', 'tracer', 'comparer', 'profil'],
    'anomaly'       : ['unusual', 'anomaly', 'anomalies', 'leak', 'strange',
                       'abnormal', 'spike', 'detect', 'suspicious', 'night',
                       'fuite', 'anomalie', 'inhabituel', 'nuit', 'détect'],
    'recommendation': ['recommend', 'suggest', 'action', 'improve', 'reduce',
                       'optimize', 'save', 'waste', 'efficiency', 'plan',
                       'recommand', 'suggér', 'réduire', 'économis', 'efficac'],
    'forecast'      : ['forecast', 'predict', 'next', 'future', 'prévision',
                       'prédire', 'prochain', 'futur'],
    'pattern'       : ['pattern', 'sequence', 'behavior', 'habit', 'markov',
                       'follow', 'after', 'flush', 'sink',
                       'séquence', 'comportement', 'habitude'],
    # 🆕 Intents contextuels
    'weather_query' : ['weather', 'temperature', 'heat', 'cold', 'season',
                       'météo', 'chaleur', 'température', 'saison', 'canicule'],
    'prayer_query'  : ['prayer', 'wudu', 'ablution', 'salat', 'prière',
                       'وضوء', 'صلاة', 'mosque', 'mosquée'],
}


def router_node(state: AgentState) -> AgentState:
    """
    Classifie l'intention de la requête.
    🆕 Priorité 1 : détection de salutation → réponse courte directe.
    🆕 Priorité 2 : intents météo et prière avant les intents génériques.
    """
    q      = state['user_query'].lower().strip()
    tokens = set(q.rstrip('!,.?').split())

    # Priorité absolue : salutation → nœud greeting
    if tokens & GREETING_WORDS or q in GREETING_WORDS:
        return {**state, 'intent': 'greeting'}

    scores = {intent: sum(1 for kw in kws if kw in q)
              for intent, kws in INTENT_KEYWORDS.items()}
    best   = max(scores, key=scores.get)
    intent = best if scores[best] > 0 else 'query'
    return {**state, 'intent': intent}


# ── Nœud : Requête & Récupération ────────────────────────────────────────────
def query_node(state: AgentState) -> AgentState:
    q         = state['user_query'].lower()
    customers = [c for c in ['A','B','C','Gym'] if c.lower() in q] or None
    sub_cats  = [s for s in ['Flush','Sink','Tap','Shower'] if s.lower() in q] or None

    data = query_data(customers=customers, sub_categories=sub_cats, agg='daily')

    data_str = data.head(20).to_string(index=False)
    hits     = semantic_search(state['user_query'], n_results=3)
    sem_ctx  = '\n'.join([h['document'][:150] for h in hits])

    prompt   = (
        f"User query: {state['user_query']}\n\n"
        f"Relevant data (daily aggregation):\n{data_str}\n\n"
        f"Similar historical records:\n{sem_ctx}\n\n"
        "Answer the query with specific numbers and water domain context."
    )
    response = call_llm(prompt)
    return {**state, 'data_context': data, 'semantic_hits': hits,
            'llm_response': response, 'final_answer': response}


# ── Nœud : Analyse & Visualisation ───────────────────────────────────────────
def analysis_node(state: AgentState) -> AgentState:
    q         = state['user_query'].lower()
    customers = [c for c in ['A','B','C','Gym'] if c.lower() in q] or ['A','B','C','Gym']

    daily = query_data(customers=customers, agg='daily')

    # Graphique de tendance
    fig = None
    if not daily.empty and 'date' in daily.columns:
        daily['date'] = pd.to_datetime(daily['date'])
        trend = daily.groupby(['customer','date'])['total'].sum().reset_index()
        fig = px.line(
            trend, x='date', y='total', color='customer',
            color_discrete_map=COLORS,
            title='Tendance de consommation journalière',
            labels={'total': 'L/jour'},
            template='plotly_white'
        )
        fig.show()

    stats_str = daily.groupby('customer')['total'].describe().round(2).to_string()
    prompt    = (
        f"User query: {state['user_query']}\n\n"
        f"Daily consumption statistics:\n{stats_str}\n\n"
        "Provide a data-driven analysis with trends, comparisons, and insights."
    )
    response  = call_llm(prompt)
    return {**state, 'data_context': daily, 'chart': fig,
            'llm_response': response, 'final_answer': response}


# ── Nœud : Détection d'Anomalies 🆕 enrichi météo + prières ─────────────────
def anomaly_node(state: AgentState) -> AgentState:
    q         = state['user_query'].lower()
    customers = [c for c in ['A','B','C','Gym'] if c.lower() in q] or ['A','B','C','Gym']

    # 🆕 Contexte météo + prières pour la date la plus récente du dataset
    try:
        latest_date = str(df_ann['timestamp'].max().date())
    except Exception:
        from datetime import date
        latest_date = str(date.today())
    weather_ctx = get_weather_context_text(latest_date)
    prayer_ctx  = get_prayer_context_text(latest_date)

    anomalies = df_ann[
        (df_ann['anomaly_label'] == 1) &
        (df_ann['customer'].isin(customers))
    ].copy()

    # 🆕 Séparation anomalies réelles vs contextuelles
    has_confidence = 'anomaly_confidence' in anomalies.columns
    if has_confidence:
        true_anom    = anomalies[anomalies['anomaly_confidence'] > 0.5].sort_values('anomaly_confidence', ascending=False)
        context_anom = anomalies[anomalies['anomaly_confidence'] <= 0.5].sort_values('anomaly_confidence', ascending=False)
    else:
        true_anom    = anomalies.sort_values('anomaly_score', ascending=False)
        context_anom = pd.DataFrame()

    top_anom = true_anom.head(50)
    normal   = df_ann[df_ann['anomaly_label'] == 0].sample(min(500, len(df_ann)))

    # 🆕 Graphique enrichi : 3 séries (Normal / Anomalie réelle / Contextuelle)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=normal['timestamp'], y=normal['consumption_L'],
        mode='markers', name='Normal',
        marker=dict(size=3, color='#90CAF9', opacity=0.4)
    ))
    fig.add_trace(go.Scatter(
        x=top_anom['timestamp'], y=top_anom['consumption_L'],
        mode='markers', name='⚠️ Anomalie réelle (conf > 0.5)',
        marker=dict(size=8, color='red', symbol='x'),
        text=top_anom.apply(
            lambda r: (
                f"Conf:{r.get('anomaly_confidence', r['anomaly_score']):.2f} | "
                f"{r['consumption_L']:.1f}L | H{r['hour']} | "
                f"{r.get('anomaly_reason', '')}"
            ), axis=1
        ),
        hovertemplate='%{text}<extra></extra>'
    ))
    if not context_anom.empty:
        fig.add_trace(go.Scatter(
            x=context_anom['timestamp'], y=context_anom['consumption_L'],
            mode='markers', name='📍 Contextuelle (météo/prière)',
            marker=dict(size=6, color='orange', symbol='circle', opacity=0.6),
            text=context_anom.apply(
                lambda r: f"{r.get('anomaly_reason','?')} | conf:{r.get('anomaly_confidence',0):.2f}",
                axis=1
            ),
            hovertemplate='%{text}<extra></extra>'
        ))
    fig.update_layout(
        title="Détection d'Anomalies — Enrichi Météo + Horaires Prière",
        yaxis_type='log', template='plotly_white'
    )
    fig.show()

    # 🆕 Prompt enrichi avec contexte météo + prière
    top5_cols = ['customer','device','timestamp','consumption_L',
                 'flow_rate_Lmin','hour','is_night','anomaly_score']
    if has_confidence:
        top5_cols += ['anomaly_confidence','anomaly_reason']
    top5 = true_anom.head(5)[top5_cols].to_string(index=False)

    prompt = (
        f"User query: {state['user_query']}\n\n"
        f"[WEATHER CONTEXT]\n{weather_ctx}\n"
        f"[PRAYER CONTEXT]\n{prayer_ctx}\n\n"
        f"True anomalies (confidence > 0.5): {len(true_anom)} detected.\n"
        f"Contextual anomalies (weather/prayer justified): {len(context_anom)}\n\n"
        f"Top 5 true anomalies:\n{top5}\n\n"
        "Explain each anomaly considering the weather and prayer context. "
        "Distinguish true anomalies from contextual consumption peaks. "
        "Recommend corrective actions for true anomalies only."
    )
    response = call_llm(prompt, max_tokens=800)
    return {**state, 'anomaly_context': anomalies, 'chart': fig,
            'llm_response': response, 'final_answer': response}


# ── Nœud : Recommandations ───────────────────────────────────────────────────
def recommendation_node(state: AgentState) -> AgentState:
    q         = state['user_query'].lower()
    customers = [c for c in ['A','B','C','Gym'] if c.lower() in q] or ['A','B','C','Gym']

    kpi = (df_ann[df_ann['customer'].isin(customers)]
           .groupby('customer').agg(
               total_L        =('consumption_L',  'sum'),
               debit_moyen    =('flow_rate_Lmin',  'mean'),
               events_nocturnes=('is_night',       'sum'),
               anomalies      =('anomaly_label',   'sum'),
           ).round(2).to_string())

    scores_str = compute_water_efficiency_score(df_ann).to_string(index=False)
    hits       = semantic_search(state['user_query'] + ' reduce waste recommend', n_results=3)
    sem_ctx    = '\n'.join([h['document'][:120] for h in hits])

    prompt = (
        f"User query: {state['user_query']}\n\n"
        f"KPIs by customer:\n{kpi}\n\n"
        f"Water efficiency scores:\n{scores_str}\n\n"
        f"Similar historical high-consumption patterns:\n{sem_ctx}\n\n"
        "Generate 5 prioritized, actionable water management recommendations. "
        "For each: state the problem, the evidence (with numbers), the action, and expected impact."
    )
    response = call_llm(prompt)
    return {**state, 'semantic_hits': hits, 'llm_response': response, 'final_answer': response}


# ── Nœud : Prévision ────────────────────────────────────────────────────────
def forecast_node(state: AgentState) -> AgentState:
    q         = state['user_query'].lower()
    customers = [c for c in ['A','B','C','Gym'] if c.lower() in q] or ['C']

    forecasts = []
    for c in customers:
        fc, fig = forecast_consumption(c, days=7)
        if fig: fig.show()
        if fc is not None:
            fc['customer'] = c
            forecasts.append(fc)

    if forecasts:
        fc_str = pd.concat(forecasts).to_string(index=False)
    else:
        fc_str = 'Pas assez de données historiques pour la prévision'

    prompt   = (
        f"User query: {state['user_query']}\n\n"
        f"7-day forecast results:\n{fc_str}\n\n"
        "Interpret the forecast and highlight any expected peaks or concerns."
    )
    response = call_llm(prompt)
    return {**state, 'llm_response': response, 'final_answer': response}


# ── Nœud : Patterns comportementaux ─────────────────────────────────────────
def pattern_node(state: AgentState) -> AgentState:
    q        = state['user_query'].lower()
    customer = next((c for c in ['A','B','C','Gym'] if c.lower() in q), 'C')

    markov = build_markov_matrix(customer, max_gap_min=5)
    if not markov.empty:
        fig = px.imshow(
            markov, color_continuous_scale='Blues', text_auto='.3f',
            title=f"Chaîne de Markov — Transitions d'usage (Customer {customer})",
            template='plotly_white'
        )
        fig.show()
        markov_str = markov.to_string()
    else:
        markov_str = 'Matrice non disponible (données insuffisantes)'

    prompt   = (
        f"User query: {state['user_query']}\n\n"
        f"Markov transition matrix for Customer {customer}:\n{markov_str}\n\n"
        "Interpret behavioral patterns and their implications for water efficiency."
    )
    response = call_llm(prompt)
    return {**state, 'llm_response': response, 'final_answer': response}


# ── 🆕 Nœud : Salutation ────────────────────────────────────────────────────
def greeting_node(state: AgentState) -> AgentState:
    """
    Réponse courte et directe aux salutations.
    Aucune donnée chargée, aucun graphique — juste une présentation en 2 lignes.
    """
    prompt = (
        f"{state['user_query']}"
    )
    # max_tokens=120 forcé via _is_greeting détecté dans call_llm
    response = call_llm(prompt, max_tokens=120)
    return {**state, 'final_answer': response}


# ── 🆕 Nœud : Requête Météo ──────────────────────────────────────────────────
def weather_query_node(state: AgentState) -> AgentState:
    """
    Répond aux questions sur l'impact de la météo sur la consommation d'eau.
    Tente de calculer la corrélation température-consommation si disponible.
    """
    try:
        latest_date = str(df_ann['timestamp'].max().date())
    except Exception:
        from datetime import date
        latest_date = str(date.today())
    weather_ctx = get_weather_context_text(latest_date)

    corr_text = ""
    if 'temp_mean' in df_ann.columns and df_ann['temp_mean'].notna().any():
        daily_t = df_ann.groupby('date').agg(
            total_L  = ('consumption_L', 'sum'),
            temp_mean= ('temp_mean',     'first'),
        ).dropna()
        if len(daily_t) > 10:
            corr = daily_t[['total_L','temp_mean']].corr().iloc[0, 1]
            corr_text = f"\nCorrélation température-consommation : r = {corr:.3f}\n"
            fig = px.scatter(daily_t, x='temp_mean', y='total_L', trendline='ols',
                             title='Corrélation Température vs Consommation journalière',
                             labels={'temp_mean': 'T° moyenne (°C)', 'total_L': 'Consommation (L)'},
                             template='plotly_white')
            fig.show()

    prompt = (
        f"User query: {state['user_query']}\n\n"
        f"[WEATHER CONTEXT]\n{weather_ctx}\n"
        f"{corr_text}"

        "Explain the relationship between ambient temperature and water consumption. "
        "Mention canicule threshold (+35°C → +40% cold water), frost risk for night leaks, "
        "and how this affects anomaly detection accuracy."
    )
    response = call_llm(prompt, max_tokens=500)
    return {**state, 'final_answer': response}


# ── 🆕 Nœud : Requête Horaires de Prière ────────────────────────────────────
def prayer_query_node(state: AgentState) -> AgentState:
    """
    Répond aux questions sur les ablutions (وضوء) et l'impact des prières.
    Analyse spécifique CustomerB qui possède des robinets وضوء.
    """
    try:
        latest_date = str(df_ann['timestamp'].max().date())
    except Exception:
        from datetime import date
        latest_date = str(date.today())
    prayer_ctx = get_prayer_context_text(latest_date)

    # Statistiques CustomerB autour des heures de prière
    b = df_ann[df_ann['customer'] == 'B'].copy()
    prayer_hours_approx = {4, 5, 12, 13, 16, 17, 19, 20, 21}
    b_prayer     = b[b['hour'].isin(prayer_hours_approx)]
    b_off_prayer = b[~b['hour'].isin(prayer_hours_approx)]
    stats_text = (
        f"CustomerB — Consommation aux heures de prière (~{len(prayer_hours_approx)} tranches):\n"
        f"  Volume total : {b_prayer['consumption_L'].sum():.1f} L\n"
        f"  Événements   : {len(b_prayer)}\n"
        f"  Moy/usage    : {b_prayer['consumption_L'].mean():.2f} L\n"
        f"vs. hors prières :\n"
        f"  Moy/usage    : {b_off_prayer['consumption_L'].mean():.2f} L\n"
        f"  Ratio        : x{b_prayer['consumption_L'].mean()/max(b_off_prayer['consumption_L'].mean(),0.01):.2f}"
    )

    prompt = (
        f"User query: {state['user_query']}\n\n"
        f"[PRAYER CONTEXT]\n{prayer_ctx}\n\n"
        f"{stats_text}\n\n"
        "Analyze wudu/ablution (وضوء) patterns. Explain how prayer times cause "
        "expected consumption spikes (~1.5L per ablution) and why these should "
        "NOT be flagged as anomalies in the system."
    )
    response = call_llm(prompt, max_tokens=500)
    return {**state, 'final_answer': response}


# ── Construction du graphe LangGraph ────────────────────────────────────────
def build_graph():
    g = StateGraph(AgentState)

    # Nœuds
    g.add_node('router',         router_node)
    g.add_node('greeting',       greeting_node)        # 🆕
    g.add_node('query',          query_node)
    g.add_node('analysis',       analysis_node)
    g.add_node('anomaly',        anomaly_node)
    g.add_node('recommendation', recommendation_node)
    g.add_node('forecast',       forecast_node)
    g.add_node('pattern',        pattern_node)
    g.add_node('weather_query',  weather_query_node)   # 🆕
    g.add_node('prayer_query',   prayer_query_node)    # 🆕

    # Routage
    g.set_entry_point('router')
    g.add_conditional_edges(
        'router',
        lambda s: s['intent'],
        {
            'greeting'      : 'greeting',              # 🆕
            'query'         : 'query',
            'analysis'      : 'analysis',
            'anomaly'       : 'anomaly',
            'recommendation': 'recommendation',
            'forecast'      : 'forecast',
            'pattern'       : 'pattern',
            'weather_query' : 'weather_query',         # 🆕
            'prayer_query'  : 'prayer_query',          # 🆕
        }
    )
    for node in ['greeting', 'query', 'analysis', 'anomaly',
                 'recommendation', 'forecast', 'pattern',
                 'weather_query', 'prayer_query']:
        g.add_edge(node, END)

    return g.compile()


agent = build_graph()
print("✅ Agent LangGraph v3 compilé | 9 nœuds :")
print("   greeting · query · analysis · anomaly · recommendation")
print("   forecast · pattern · weather_query · prayer_query")

✅ Agent LangGraph v3 compilé | 9 nœuds :
   greeting · query · analysis · anomaly · recommendation
   forecast · pattern · weather_query · prayer_query


---
## 14. 🚀 Démo Live — Exécution de l'Agent

In [47]:
def run_agent(query: str, verbose: bool = True) -> dict:
    """
    Exécute le pipeline complet de l'agent WaterSec.
    Retourne l'état final de l'agent.
    """
    if verbose:
        print('─' * 72)
        print(f'🔵 Requête : {query}')
        print('─' * 72)

    t0 = time.time()
    initial_state: AgentState = {
        'user_query'     : query,
        'intent'         : '',
        'data_context'   : None,
        'anomaly_context': None,
        'semantic_hits'  : None,
        'chart'          : None,
        'llm_response'   : '',
        'final_answer'   : '',
        'context_info'   : '',   # 🆕 rempli par les nœuds contextuels
    }

    result  = agent.invoke(initial_state)
    elapsed = time.time() - t0

    if verbose:
        print(f'\n🟢 Intent détecté : {result["intent"]} | Temps : {elapsed:.1f}s')
        print()
        print(textwrap.fill(result['final_answer'], width=80))
        print()

    return result

In [48]:
# ── Prompt 1 : Requête simple ─────────────────────────────────────────────────
r1 = run_agent(
    "What is the average daily cold water consumption in Customer B for the last 30 days?"
)

────────────────────────────────────────────────────────────────────────
🔵 Requête : What is the average daily cold water consumption in Customer B for the last 30 days?
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : query | Temps : 0.9s

1 chiffre : 27,64 La consommation moyenne quotidienne d'eau froide pour le
Customer B n'est pas directement disponible, mais en considérant les données
similaires du Customer C, on peut estimer que la consommation moyenne
quotidienne d'eau froide pour le Customer B pourrait être d'environ 27,64 litres
par jour, en se basant sur les données historiques du Customer C.



In [49]:

# ── Prompt 2 : Analyse & Visualisation ───────────────────────────────────────
r2 = run_agent(
    "Plot the daily consumption trend for all customers and compare their profiles"
)

────────────────────────────────────────────────────────────────────────
🔵 Requête : Plot the daily consumption trend for all customers and compare their profiles
────────────────────────────────────────────────────────────────────────



🟢 Intent détecté : analysis | Temps : 1.0s

📊 Les tendances de consommation d'eau quotidienne varient considérablement entre
les clients. 🔢 Client A : 233,22 L/jour, Client C : 44,70 L/jour. ✅ Le Client A
présente une consommation d'eau plus élevée et plus variable que le Client C,
avec une moyenne quotidienne 5 fois supérieure.  À surveiller : Les pics de
consommation du Client A pour détecter d'éventuelles fuites ou gaspillages.



In [60]:
# ── Prompt 3 : Détection d'anomalies ─────────────────────────────────────────
r3 = run_agent(
    "detecter les anomalies du costumer A pendant le matin "
)

────────────────────────────────────────────────────────────────────────
🔵 Requête : detecter les anomalies du costumer A pendant le matin 
────────────────────────────────────────────────────────────────────────



🟢 Intent détecté : anomaly | Temps : 1.3s

📊 Les anomalies détectées pour le Customer A doivent être analysées pour
identifier les consommations anormales pendant le matin. 🔢 L'anomalie du
2025-08-09 11:22:15+00:00 montre une consommation de 500 L en une session, ce
qui est anormalement élevé. ✅ Cette anomalie est une vraie anomalie (confiance >
0,5) et nécessite une action corrective. À surveiller : Les habitudes de
consommation d'eau du Customer A pour détecter d'éventuelles fuites ou
anomalies. Recommandations : 1. Vérifier les installations d'eau du Customer A
pour détecter d'éventuelles fuites. 2. Mettre en place un système de
surveillance pour détecter les anomalies de consommation d'eau pendant le matin.
3. Éduquer les utilisateurs sur les bonnes pratiques de conservation de l'eau
pour minimiser les gaspillages.



In [54]:
# ── Prompt 4 : Patterns comportementaux (niveau avancé) ───────────────────────
r4 = run_agent(
    "Identify recurrent usage patterns in Customer C. Does a toilet flush often lead to sink usage? Detect and describe such patterns."
)

────────────────────────────────────────────────────────────────────────
🔵 Requête : Identify recurrent usage patterns in Customer C. Does a toilet flush often lead to sink usage? Detect and describe such patterns.
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : pattern | Temps : 10.0s

📊 Les données pour le Customer C montrent que les événements de flush des
toilettes sont souvent suivis d'une utilisation des sinks. 🔢 Probabilité de 0,8
que l'utilisation du sink suive un flush de toilette. ✅ Cette corrélation
suggère que les habitudes des utilisateurs incluent souvent une ablution après
l'utilisation des toilettes, ce qui est cohérent avec les pratiques d'hygiène. À
surveiller : Les dispositifs d'eau du Customer C pour optimiser la consommation
d'eau en fonction de ces habitudes.  Recommandations : 1. Installer des
dispositifs d'eau économes pour les sinks. 2. Mettre en place un système de
surveillance pour détecter les anomalies de consomm

In [52]:
# ── Prompt 5 : Prévision ──────────────────────────────────────────────────────
r5 = run_agent(
    "Forecast the water consumption for the next 7 days for Customer C and Gym"
)

DEBUG:cmdstanpy:input tempfile: /tmp/tmp4ejl2fjt/6dp6f0n1.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmp4ejl2fjt/y2pc4lia.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=50297', 'data', 'file=/tmp/tmp4ejl2fjt/6dp6f0n1.json', 'init=/tmp/tmp4ejl2fjt/y2pc4lia.json', 'output', 'file=/tmp/tmp4ejl2fjt/prophet_modeloalm0moh/prophet_model-20260510100226.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
10:02:26 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
10:02:26 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


────────────────────────────────────────────────────────────────────────
🔵 Requête : Forecast the water consumption for the next 7 days for Customer C and Gym
────────────────────────────────────────────────────────────────────────


DEBUG:cmdstanpy:input tempfile: /tmp/tmp4ejl2fjt/dsr0rypv.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmp4ejl2fjt/gzv67u6h.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=85185', 'data', 'file=/tmp/tmp4ejl2fjt/dsr0rypv.json', 'init=/tmp/tmp4ejl2fjt/gzv67u6h.json', 'output', 'file=/tmp/tmp4ejl2fjt/prophet_model104cnp_o/prophet_model-20260510100227.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
10:02:27 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
10:02:27 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


DEBUG:cmdstanpy:input tempfile: /tmp/tmp4ejl2fjt/5te0vnon.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmp4ejl2fjt/kn42h8o0.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=99163', 'data', 'file=/tmp/tmp4ejl2fjt/5te0vnon.json', 'init=/tmp/tmp4ejl2fjt/kn42h8o0.json', 'output', 'file=/tmp/tmp4ejl2fjt/prophet_modelnt_70_u9/prophet_model-20260510100227.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
10:02:27 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
10:02:27 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


⏳ Rate limit — retry dans 5s...

🟢 Intent détecté : forecast | Temps : 6.2s

❌ Erreur LLM : Error code: 413 - {'error': {'message': 'Request too large for
model `llama-3.3-70b-versatile` in organization `org_01kr6jr35bfhmvvn87g9566my9`
service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested
38668, please reduce your message size and try again. Need more tokens? Upgrade
to Dev Tier today at https://console.groq.com/settings/billing', 'type':
'tokens', 'code': 'rate_limit_exceeded'}}



In [53]:
# ── Prompt 6 : Recommandations (niveau avancé) ────────────────────────────────
r6 = run_agent(
    "Based on all data, recommend the top 5 actions to reduce water waste across all sites. Include evidence and expected impact."
)

────────────────────────────────────────────────────────────────────────
🔵 Requête : Based on all data, recommend the top 5 actions to reduce water waste across all sites. Include evidence and expected impact.
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : recommendation | Temps : 1.6s

1. **Réduire les fuites nocturnes chez Customer B** :     - Problème : Fuites
nocturnes importantes.    - Évidence : 1291 événements nocturnes, score nocturne
de 57,9.    - Action : Vérifier et réparer les installations d'eau.    - Impact
attendu : Réduction de 20% de la consommation d'eau nocturne.  2. **Optimiser
les dispositifs d'eau chez Customer A** :    - Problème : Débit moyen élevé.
- Évidence : Débit moyen de 4,79 L/min, score débit de 80,0.    - Action :
Installer des dispositifs d'eau économes.    - Impact attendu : Réduction de 15%
de la consommation d'eau.  3. **Améliorer la régularité de la consommation d'eau
chez Customer C** :    - Problème :

In [59]:
# ── Prompt 7 : Comparaison cabines Gym ───────────────────────────────────────
r7 = run_agent(
    "Compare shower cabin 1 and cabin 2 in the gym — which uses more hot water?"
)

────────────────────────────────────────────────────────────────────────
🔵 Requête : Compare shower cabin 1 and cabin 2 in the gym — which uses more hot water?
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : query | Temps : 23.6s

📊 Les données pour les cabines de douche du gymnase montrent que la cabine 1
utilise en moyenne 12,51 L d'eau chaude par usage, tandis que la cabine 2
utilise en moyenne 10,35 L d'eau chaude par usage. 🔢 La cabine 1 utilise 20,5%
plus d'eau chaude que la cabine 2. ✅ La cabine 1 est la plus grande
consommatrice d'eau chaude, avec un total de 245,1 L d'eau chaude utilisée sur
la période étudiée. À surveiller : Les habitudes de consommation d'eau des
cabines de douche pour optimiser la consommation d'eau chaude. Recommandations :
1. Vérifier les installations d'eau chaude des cabines de douche pour détecter
d'éventuelles fuites. 2. Mettre en place un système de surveillance pour
détecter les anomalies de consommation 

In [58]:
# ── Prompt 8 : Fuite nocturne ─────────────────────────────────────────────────
r8 = run_agent(
    "Detect any night leaks or unusual nocturnal water flow across all customers. Which device is at highest risk?"
)

────────────────────────────────────────────────────────────────────────
🔵 Requête : Detect any night leaks or unusual nocturnal water flow across all customers. Which device is at highest risk?
────────────────────────────────────────────────────────────────────────



🟢 Intent détecté : anomaly | Temps : 13.3s

📊 Les anomalies détectées doivent être analysées pour identifier les fuites
nocturnes ou les flux d'eau inhabituels. 🔢 Le device
11d8a970-daf7-11ee-b288-f5d6bb92dfd4 du Customer C présente un débit nocturne
suspect de 4,70 L/min le 2024-07-27 05:47:31+00:00. ✅ Ce device est à haut
risque de fuite nocturne en raison de son débit élevé pendant la nuit. À
surveiller : Les habitudes de consommation d'eau du Customer C pour détecter
d'éventuelles fuites nocturnes. Recommandations : 1. Vérifier les installations
d'eau du device 11d8a970-daf7-11ee-b288-f5d6bb92dfd4 pour détecter d'éventuelles
fuites. 2. Mettre en place un système de surveillance pour détecter les
anomalies de consommation d'eau nocturne. 3. Éduquer les utilisateurs sur les
bonnes pratiques de conservation de l'eau pour minimiser les gaspillages.



---
## 14b. 🆕 Tests des nouveaux nœuds — Météo, Prières, Salutations


In [64]:
# ── Test salutation courte ────────────────────────────────────────────────────
# Attendu : réponse en 1-2 lignes, PAS de données ou de tableaux
r_hi = run_agent("hi")


────────────────────────────────────────────────────────────────────────
🔵 Requête : hi
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : greeting | Temps : 29.7s

Bonjour ! Comment puis-je vous aider aujourd'hui ?



In [71]:
# ── Test salutation en arabe ─────────────────────────────────────────────────
r_salam = run_agent("السلام عليكم")


────────────────────────────────────────────────────────────────────────
🔵 Requête : السلام عليكم
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : greeting | Temps : 0.3s

وعليكم السلام ورحمة الله وبركاته



In [55]:
# ── Test impact météo sur consommation ───────────────────────────────────────
r_weather = run_agent(
    "How does hot weather and high temperature affect water consumption at the gym?"
)


────────────────────────────────────────────────────────────────────────
🔵 Requête : How does hot weather and high temperature affect water consumption at the gym?
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : weather_query | Temps : 5.3s

📊 Les températures élevées ont un impact sur la consommation d'eau au gymnase. 🔢
Au-dessus de 35°C, la consommation d'eau froide augmente de 40%. ✅ Les canicules
entraînent une augmentation de la consommation d'eau pour la douche et les
activités sportives, tandis que les périodes de froid peuvent augmenter le
risque de fuites nocturnes dues au gel. À surveiller : Les anomalies de
consommation d'eau pendant les périodes de canicule ou de froid pour éviter les
faux positifs.  Recommandations : 1. Ajuster les seuils d'anomalie en fonction
de la température ambiante. 2. Mettre en place un système de surveillance pour
détecter les fuites nocturnes pendant les périodes de froid. 3. Éduquer les
utilisateurs su

In [56]:
# ── Test horaires de prière — CustomerB ──────────────────────────────────────
r_prayer = run_agent(
    "Analyse the wudu and ablution patterns for Customer B. "
    "Are the consumption spikes around prayer times normal?"
)


────────────────────────────────────────────────────────────────────────
🔵 Requête : Analyse the wudu and ablution patterns for Customer B. Are the consumption spikes around prayer times normal?
────────────────────────────────────────────────────────────────────────

🟢 Intent détecté : prayer_query | Temps : 15.6s

📊 Les données pour le Customer B montrent des pics de consommation d'eau autour
des heures de prière. 🔢 La consommation moyenne par usage est de 12,58 L pendant
les heures de prière, contre 11,41 L hors prières. ✅ Ces pics de consommation
sont normaux en raison des ablutions (~1,5 L par ablution) effectuées avant les
prières, et ne devraient pas être signalés comme des anomalies dans le système.
À surveiller : Les habitudes de consommation d'eau du Customer B pour détecter
d'éventuelles anomalies hors des heures de prière.  Recommandations : 1. Exclure
les heures de prière de l'analyse des anomalies pour éviter les faux positifs.
2. Mettre en place un système de surveillanc

In [57]:
# ── Test anomalie enrichie météo+prière ──────────────────────────────────────
r_anom_ctx = run_agent(
    "Is there any unusual behaviour on Customer B during evening hours? "
    "Are these anomalies real or linked to prayer times?"
)


────────────────────────────────────────────────────────────────────────
🔵 Requête : Is there any unusual behaviour on Customer B during evening hours? Are these anomalies real or linked to prayer times?
────────────────────────────────────────────────────────────────────────



🟢 Intent détecté : anomaly | Temps : 12.5s

📊 Les anomalies détectées pour le Customer B doivent être analysées dans le
contexte des heures de prière et de la météo. 🔢 Les pics de consommation d'eau
pendant les heures de prière (±20 min) sont normaux en raison des ablutions
(~1,5 L par ablution). ✅ Les vraies anomalies (confiance > 0,5) doivent être
distinguées des pics de consommation contextuels liés aux heures de prière ou à
la météo. À surveiller : Les habitudes de consommation d'eau du Customer B pour
détecter d'éventuelles anomalies hors des heures de prière. Recommandations pour
les vraies anomalies : 1. Vérifier les installations d'eau pour détecter
d'éventuelles fuites (anomalie du 2026-04-05 04:05:53+00:00). 2. Analyser les
données de consommation d'eau pour comprendre les causes de l'anomalie du
2024-03-23 06:59:30+00:00. 3. Mettre en place un système de surveillance pour
détecter les anomalies de consommation d'eau hors des heures de prière.



---
## 15. 📤 Export des Artefacts

In [ ]:
import os

OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

# ── Export 1 : Dataset enrichi + annoté ──────────────────────────────────────
export_cols = [
    'customer', 'device', 'timestamp', 'consumption_L', 'data_period',
    'flow_rate_Lmin', 'main_category_name', 'sub_category_name', 'tag',
    'hour', 'dayofweek', 'is_weekend', 'is_night', 'month', 'week',
    'anomaly_label', 'anomaly_score', 'anomaly_method'
]
df_ann[export_cols].to_csv(f'{OUT_DIR}/watersec_enriched_annotated.csv', index=False)

# ── Export 2 : Anomalies uniquement ──────────────────────────────────────────
anomalies_export = df_ann[df_ann['anomaly_label'] == 1][export_cols]
anomalies_export.to_csv(f'{OUT_DIR}/watersec_anomalies.csv', index=False)

# ── Export 3 : KPIs journaliers ───────────────────────────────────────────────
daily_export = (df_ann.groupby(['customer','device','date']).agg(
    total_L              =('consumption_L',  'sum'),
    events               =('consumption_L',  'count'),
    avg_flow             =('flow_rate_Lmin', 'mean'),
    night_events         =('is_night',       'sum'),
    anomaly_count        =('anomaly_label',  'sum'),
    max_anomaly_score    =('anomaly_score',  'max'),
).reset_index())
daily_export.to_csv(f'{OUT_DIR}/watersec_daily_kpis.csv', index=False)

# ── Export 4 : Scores d'efficacité ───────────────────────────────────────────
df_scores.to_csv(f'{OUT_DIR}/watersec_efficiency_scores.csv', index=False)

# ── Export 5 : Fuites nocturnes ───────────────────────────────────────────────
if not df_leaks.empty:
    df_leaks.to_csv(f'{OUT_DIR}/watersec_night_leaks.csv', index=False)

print(f'\n✅ Exports sauvegardés dans : {OUT_DIR}/')
print(f'  📄 watersec_enriched_annotated.csv  → {len(df_ann):,} lignes')
print(f'  📄 watersec_anomalies.csv            → {len(anomalies_export):,} lignes')
print(f'  📄 watersec_daily_kpis.csv           → {len(daily_export):,} lignes')
print(f'  📄 watersec_efficiency_scores.csv    → {len(df_scores)} clients')
if not df_leaks.empty:
    print(f'  📄 watersec_night_leaks.csv          → {len(df_leaks)} fuites détectées')

---
## 📐 Résumé Architecture v3

```
╔══════════════════════════════════════════════════════════════════════════╗
║         WaterSec AI Agent v3 — Architecture Enrichie                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║  COUCHE 1 — DONNÉES                                                     ║
║  ┌──────────────────┐  ┌───────────────────────────────────────────┐   ║
║  │  pandas SQL-like  │  │  ChromaDB + BGE-small embeddings          │   ║
║  │  query_data()     │  │  Documents journaliers enrichis           │   ║
║  └──────────────────┘  └───────────────────────────────────────────┘   ║
╠══════════════════════════════════════════════════════════════════════════╣
║  COUCHE 2 — INTELLIGENCE DOMAINE EAU                                   ║
║  • Water Demand Index (WDI) — profil normalisé inter-clients            ║
║  • Score d'efficacité hydrique [0-100] — 4 composantes                 ║
║  • Détection fuites continues — débit nocturne persistant               ║
║  • Analyse Markov — transitions comportementales entre usages           ║
║  • Prévision Prophet — 7 jours avec intervalles de confiance            ║
╠══════════════════════════════════════════════════════════════════════════╣
║  COUCHE 2b — 🆕 CONTEXTE EXTERNE                                       ║
║  • Open-Meteo API — température / précipitations / humidité             ║
║    → Canicule >35°C : +40% eau froide attendu (non-anomalie)           ║
║    → Gel <5°C       : risque fuites nocturnes accru                    ║
║  • Aladhan API — horaires Fajr/Dhuhr/Asr/Maghrib/Isha                 ║
║    → Pics CustomerB ±20 min des prières = ablution (وضوء) normale     ║
╠══════════════════════════════════════════════════════════════════════════╣
║  COUCHE 3 — ML ANOMALIES ENRICHIES                                     ║
║  • Isolation Forest (200 arbres, contamination=3%)                     ║
║  • Z-Score adaptatif (fenêtre glissante 24h, σ=3)                      ║
║  • 🆕 Score de confiance contextuel [0-1]                              ║
║    - Réduit de 75% si événement coïncide avec un salat                 ║
║    - Réduit de 20-40% si température justifie la consommation          ║
║  • 🆕 anomaly_reason — explication textuelle par anomalie              ║
╠══════════════════════════════════════════════════════════════════════════╣
║  COUCHE 4 — ORCHESTRATEUR LangGraph v3                                 ║
║  router → [greeting | query | analysis | anomaly | recommendation |    ║
║             forecast | pattern | weather_query | prayer_query]         ║
║  🆕 greeting : réponse 2 lignes max, 120 tokens, pas de données        ║
║  🆕 weather_query : corrélation T° ↔ consommation + graphique          ║
║  🆕 prayer_query  : stats ablution CustomerB + interprétation          ║
╠══════════════════════════════════════════════════════════════════════════╣
║  COUCHE 5 — LLM REASONING (GROQ — GRATUIT)                            ║
║  • llama-3.3-70b-versatile via groq.com                                ║
║  • 🆕 System prompt v3 : règles de concision + domaine eau enrichi    ║
║    (météo, ablution, benchmarks contextuels)                           ║
║  • 🆕 max_tokens adaptatif : 120 (salut) → 800 (anomalie complexe)    ║
║  • Mémoire conversationnelle (5 derniers échanges)                     ║
╚══════════════════════════════════════════════════════════════════════════╝
```

### 🔑 Activer le LLM Groq (gratuit)
1. Créer un compte → **https://console.groq.com** (gratuit)
2. Générer une clé API (`gsk_...`)
3. Dans la cellule de config : remplacer `'gsk_VOTRE_CLE_API_ICI'`
4. Relancer → analyses LLM réelles activées

**Limites gratuites Groq :** 30 req/min · 6 000 req/jour — largement suffisant pour le hackathon